# Spectabular Visualization Report

**Nasir Khimani**  
Department of Computing and Software, McMaster University  
July 26th, 2026

## 1. Introduction

Formal specification and verification tools often provide mechanisms for executing, simulating, or exploring models, but the way these behaviours are presented to users varies considerably between tools.
Some approaches expose execution traces and variable states directly, while others provide domain specific visualizations that allow users to interact with a representation of the system being modelled.

This report explores existing approaches to visualization and animation in formal methods with the goal of identifying techniques that could support visualization in Spectabular, a Jupyter based environment for formal specification using tables.
The report first examines several formal development approaches and the visualization or animation tools associated with them, including B and Event-B with ProB and VisB, PVS with PVSio-web, Event-B with BMotion Studio, TLA+ with Spectacle, Electrum, and Simulink/Stateflow.
Other formal approaches that provide simulation without domain specific visualization are then considered.

A set of representative case studies is used to identify the kinds of behaviours a Spectabular visualization system should support, including interactive device interfaces, moving entities, concurrent behaviour, timed systems, scenario replay, and internal system state.
The existing approaches are then compared with respect to these requirements.

The main conclusion is that Spectabular should support both interactive exploration, where users directly influence model execution, and scripted animation, where predefined or recorded executions can be replayed consistently.
Both should use the same underlying visualization and remain driven by the formal specification state, while the visualization layer determines how each state and transition is presented.

Jupyter provides a useful environment for this approach because the specification, documentation, controls, and visualization can be kept together in a single notebook that can be shared with developers, regulators, or domain experts.

## 2. Classical B and Event-B with ProB and VisB

**Resources:**

- [*Modelling and Validating an Automotive System in Classical B and Event-B*](https://doi.org/10.1007/978-3-030-48077-6_27) 
- [*VisB: A Lightweight Tool to Visualize Formal Models with SVG Graphics*](https://doi.org/10.1007/978-3-030-48077-6_21)
- [VisB automotive case study repository](https://github.com/hhu-stups/abz2020-models)

Spectabular already includes an implementation of the automotive lighting case study, making this case study a useful starting point for examining existing visualization approaches.
Rather than considering visualization techniques from scratch, implementations of the same system in other formal methods can be compared with the Spectabular model.

In work by Leuschel et al., the automotive lighting system is modeled in Classical B and Event-B, using ProB and VisB for validation and visualization.
Their model covers the vehicle's exterior lighting controls, including the ignition, turn signals, hazard switch, and timing behavior, making its scope directly comparable to Spectabular's.

Classical B and Event-B are state based formal methods, but they support different styles of system development.
Classical B was originally developed with software development in mind and provides strong structuring mechanisms, including machine inclusion and operation calls.
Event-B was developed more specifically for systems modelling, with a simplified language and a more flexible refinement mechanism intended to make refinement proofs easier to manage and scale through a linear refinement chain.

These differences directly influenced the development methodology used by Leuschel et al.
The authors divide development into an exploratory phase, a synthesis phase, and a final verification phase.
Classical B is used during the first two phases because its rich substitution language, machine inclusion, and operation calls make it easier to modify individual components and experiment with different ways of composing them.
During this stage, ProB is used extensively for animation, trace replay, and model checking while the structure of the system is still changing.

Once the model has stabilized, it is translated into Event-B and arranged as a linear refinement hierarchy for the final proof phase in Rodin.
This translation is not required for visualization; ProB supports both Classical B and Event-B, and the same VisB visualization is used with both versions of the model.
Instead, the translation is a methodological choice that combines the flexibility of Classical B during exploratory modelling with the simpler proof obligations and proving environment provided by Event-B and Rodin.

![l1](./assets/image18.png)
*Machines and events in the three modelling phases. Source: VisB: A Lightweight Tool to Visualize Formal Models with SVG Graphics.*

This approach also introduces a trade off.
Classical B machine inclusion and operation calls do not translate directly into Event-B, so the component structure must be converted into a linear refinement hierarchy and some operations must be split into several events.
The resulting Event-B model contains more events and duplicated logic, particularly when timing behaviour is introduced, making later changes more tedious and error prone.
For this reason, the authors delay the conversion until the model has stabilized.
They identify the manual translation itself as a weakness of the approach, but consider the additional effort worthwhile because of the proof support available in Rodin.

The Classical B model is divided into separate components for sensors, blinking control, and time.
The `Sensors` component contains the input state, including `pitmanArmUpDown`, `keyState`, `engineOn`, and `hazardWarningSwitchOn`.
The `BlinkLamps` component contains the output actuators `blinkLeft` and `blinkRight`, together with the controller variables `active_blinkers`, `remaining_blinks`, and `onCycle`.

A simplified excerpt of the blinking component is:

```b
MACHINE BlinkLamps

INVARIANT
    active_blinkers <: BLINK_DIRECTION &
    remaining_blinks : BLINK_CYCLE_COUNTER &
    blinkLeft : LAMP_STATUS &
    blinkRight : LAMP_STATUS &
    onCycle : BOOL

OPERATIONS
    SET_BlinkersOn(direction, rem) =
        PRE
            direction : BLINK_DIRECTION &
            rem : BLINK_CYCLE_COUNTER &
            rem /= 0
        THEN
            active_blinkers := {direction} ||
            remaining_blinks := rem ||
            IF direction = right_blink THEN
                blinkLeft := lamp_off ||
                blinkRight := cycleMaxLampStatus(onCycle)
            ELSE
                blinkLeft := cycleMaxLampStatus(onCycle) ||
                blinkRight := lamp_off
            END
        END
END
```

The distinction between controller state and actuator state is important for understanding the visualization.
The set `active_blinkers` records which side the controller intends to operate, whereas `blinkLeft` and `blinkRight` record the actual state of the light actuators.

The `PitmanController` reacts to changes in the sensor state by calling operations from the sensor and blinking components.
For example, turning on the engine also starts continuous blinking when the pitman arm is in a blinking position and the hazard warning switch is off:

```b
ENV_Turn_EngineOn =
    BEGIN
        SET_EngineOn ||
        IF pitmanArmUpDown : PITMAN_DIRECTION_BLINKING &
           hazardWarningSwitchOn = switch_off
        THEN
            SET_BlinkersOn(
                pitman_direction(pitmanArmUpDown),
                continuousBlink
            )
        END
    END
```

VisB connects these model variables to named objects in an SVG using a JSON glue file.
For example, the following entry uses both the controller variable `active_blinkers` and the actuator variable `blinkRight` to determine the colour of the graphical object `A-right`:

```json
{
  "id": "A-right",
  "attr": "fill",
  "value": "IF right_blink:active_blinkers THEN IF blinkRight=lamp_off THEN \"#ffe6cc\" ELSE \"orange\" END ELSE \"white\" END"
}
```

If the controller considers the right blinkers active but the actuator is currently off, the light is displayed in a pale orange.
If the controller considers the side active and the actuator is on, it is displayed in orange.
Otherwise, it is displayed in white.
This allows the visualization to distinguish the controller's intended blinking state from the current phase of the physical blinking cycle.

Sensor values are mapped in the same way.
For example, the state of the hazard warning switch controls the opacity of the hazard warning symbol:

```json
{
  "id": "warningLight",
  "attr": "fill-opacity",
  "value": "IF hazardWarningSwitchOn=switch_on THEN \"0.7\" ELSE \"0.05\" END"
}
```

The pitman arm sensor similarly controls the emphasis applied to the corresponding graphical position:

```json
{
  "id": "PitmanUpward",
  "attr": "fill-opacity",
  "value": "IF pitmanArmUpDown=Upward7 THEN \"1\" ELSIF pitmanArmUpDown=Upward5 THEN \"0.85\" ELSE \"0.2\" END"
}
```

The mappings can be summarized as follows:

| Formal state | Role in the model | Visual representation |
|---|---|---|
| `keyState` | Ignition sensor | Visibility and colour of the key-position elements |
| `engineOn` | Engine state | Engine indicator, window colour, and car outline opacity |
| `hazardWarningSwitchOn` | Hazard switch sensor | Opacity of the hazard warning symbol |
| `pitmanArmUpDown` | Pitman arm sensor | Emphasis of the neutral, upward, or downward arm position |
| `active_blinkers` | Intended controller output | Pale highlighting and reduced stroke opacity on the selected side |
| `blinkLeft`, `blinkRight` | Light actuators | Orange fill when the corresponding lamps are on |
| `curTime` | Elapsed model time | Text displayed in the timer element |

Time is modelled using a separate generic timer component.
The variable `curTime` stores elapsed time in milliseconds, while `curDeadlines` maps individual timers to their deadlines:

```b
MACHINE GenericTimers(TIMERS)

INVARIANT
    curTime : NATURAL &
    curDeadlines : TIMERS +-> NATURAL

OPERATIONS
    AddDeadline(timer, deadline) =
        PRE
            timer : TIMERS &
            deadline : NATURAL
        THEN
            curDeadlines(timer) := curTime + deadline
        END;

    IncreaseTime(delta) =
        SELECT
            delta : NATURAL &
            (curDeadlines /= {} =>
                curTime + delta <= min(ran(curDeadlines)))
        THEN
            curTime := curTime + delta
        END
END
```

The deadline condition prevents time from advancing beyond the next scheduled event.
The corresponding blinking or tip blinking operation must therefore execute before time can continue.
For example, beginning tip blinking adds a deadline 500 ms after the current time:

```b
ENV_Pitman_Tip_blinking_start(newPos) =
    SELECT
        newPos : PITMAN_TIP_BLINKING &
        newPos /= pitmanArmUpDown
    THEN
        ENV_Pitman_Tip_blinking_short(
            newPos,
            pitman_direction(newPos)
        ) ||
        AddDeadline(tip_deadline, 500)
    END
```

The timed version of the VisB glue file can display `curTime` directly in the SVG.

The glue file can also map interactions in the opposite direction, allowing an SVG element such as the ignition control to execute an operation such as `ENV_Turn_EngineOn`.
```json 
{
    "id":"key-on-position",
    "event":"ENV_Turn_EngineOn"
},
```

The visualization therefore exposes environmental inputs, internal controller decisions, and actuator outputs in the same graphical representation.
This is more informative than mapping each light directly to a single Boolean value because inconsistencies between the controller's intended state and the physical output state can become visually apparent.

![](assets/image11.png)
*Architecture of the VisB visualization approach. Source: VisB: A Lightweight Tool to Visualize Formal Models with SVG Graphics.*

Loading the automotive lighting model in ProB shows the operations that are currently enabled alongside the VisB visualization.

![image10.png](assets/image10.png)

Executing an operation changes the model state, which can enable new operations and update the corresponding graphical elements.

For example, the engine can first be turned on and time advanced by 500ms.

![image1.png](assets/image1.png)

![image6.png](assets/image6.png)

Because time is represented explicitly in the model, the execution can be followed one transition at a time while observing how the visualization changes.

The sequence above produces an operation trace in ProB containing both environment actions and timed transitions.
This allows a scenario to be replayed while viewing the corresponding graphical state at each step.

VisB therefore provides a two-way connection between the graphical representation and the formal model.
Formal expressions in the JSON glue file determine properties such as colour, opacity, visibility, and text, while graphical elements can also be associated with operations that are executed through ProB.

The visualization remains separate from the formal specification itself.
The same model can therefore still be animated, model checked, and proved using the existing B and Event-B toolchain, while VisB provides a domain specific interface for exploring its behaviour.

## 3. PVS and PVSio-web

**Resources:**

- [*PVSio-web: A Tool for Rapid Prototyping Device User Interfaces in PVS*](https://doi.org/10.14279/tuj.eceasst.69.963)
- [*PVSio-web: Mathematically Based Tool Support for the Design of Interactive and Interoperable Medical Systems*](https://doi.org/10.4108/eai.14-10-2015.2261720)
- [*Extending a User Interface Prototyping Tool with Automatic MISRA C Code Generation*](https://doi.org/10.4204/EPTCS.240.4)
- [PVS documentation](https://pvs.csl.sri.com/documentation.html)

Another approach to connecting formal specifications with domain specific visualizations is PVSio-web.
Unlike VisB, which provides a visualization of a B or Event-B model being explored through ProB, PVSio-web is designed around interactive device interfaces.
The user interacts with a graphical representation of a device while the behaviour of that interface is determined by an underlying formal specification.

PVS is a formal specification and verification system based on typed higher order logic.
Specifications are organized into theories containing types, constants, functions, and properties to be proved.
PVS includes a type checker and an interactive theorem prover, allowing properties of a specification to be formally verified.
It also provides symbolic model checking capabilities and can generate executable code from parts of a specification.

PVSio provides an execution environment for PVS specifications.
Executable functions in a specification can be evaluated interactively, allowing a formal model to be used as a simulator.
Without an additional interface, this interaction is primarily textual; functions are executed by entering PVS expressions and the resulting state is returned as structured text.

PVSio-web places a graphical interface in front of this execution environment.
A designer can load an image of a device and define interactive regions over elements such as buttons and displays.
These regions are then connected to functions and state values in the PVS specification.
The graphical prototype therefore acts as an interface to the underlying formal model rather than implementing the behaviour independently.

The approach is demonstrated using the interface of a commercial medical infusion pump.
The prototype begins with an image of the device, over which the designer defines regions corresponding to buttons and displays.
These regions are then connected to elements of the PVS specification.

A simple example is the numeric display and the UP button used to enter an infusion value.
The value shown on the device is represented by a field called `display` in the PVS specification.
PVSio-web places a display region over the corresponding part of the device image and associates it with this field.
When PVSio returns a new formal state, PVSio-web extracts the value of `display` and renders it in this region.

The connection also works in the opposite direction.
A region placed over the graphical UP button is associated with the PVS functions `press_UP` and `release_UP`.
Pressing the graphical button causes `press_UP` to be executed by PVSio, while releasing it executes `release_UP`.
For example, when the initial display value is `0`, clicking the UP button executes these functions and produces a new formal state in which the display value is `1`.
PVSio-web then extracts this new value and updates the graphical display.

![image20.png](assets/image20.png)
*The graphical environment of PVSio-web. Source: PVSio-web: A Tool for Rapid Prototyping Device User Interfaces in PVS*

For press and hold interactions, `press_UP` can be executed repeatedly while the button remains pressed, with each resulting formal state being returned to the interface.
This allows the displayed value to increase continuously while the behaviour remains defined by the PVS specification rather than being implemented independently in the graphical interface.

The connection can therefore be summarized as follows:

| PVS element | Role | PVSio-web representation |
|---|---|---|
| `display` | Current numeric value in the formal state | Numeric display region on the device image |
| `press_UP` | State transition caused by pressing the UP button | Executed when the graphical UP button is pressed |
| `release_UP` | State transition caused by releasing the UP button | Executed when the graphical UP button is released |
| Resulting PVS state | New formal state after an interaction | Parsed to update the visible display |

This creates a two-way connection between the formal specification and the visualization.
User interaction with a graphical region is translated into a function call in the PVS specification, while values from the resulting formal state are translated back into changes in the graphical interface.

The architecture separates the graphical front end from the formal execution backend.
The browser based front end provides the interface builder and simulator, while the backend runs PVS and PVSio.
User interactions are sent to the backend as commands for PVSio, and the resulting formal state is returned to the front end so that the corresponding display regions can be updated.

![image19.png](assets/image19.png)
*Architecture of PVSio-web. Source: PVSio-web: A Tool for Rapid Prototyping Device User Interfaces in PVS*

Because the behaviour remains represented as a PVS specification, the same model used to drive the prototype can also be analysed using the verification facilities available in PVS.
Properties can be proved using the PVS theorem prover, while executable portions of the specification can be explored through PVSio.
PVS also supports code generation from executable specifications, and later work extended the PVSio-web workflow with generation of MISRA C code from formally modelled user interface behaviour.

PVSio-web therefore demonstrates a different style of domain specific visualization from VisB.
Instead of primarily visualizing the internal state of a formal system, it creates a realistic interactive prototype through which the formal model can be operated.
This is particularly useful for systems in which the user interface itself is an important part of the behaviour being validated, such as medical devices with buttons, displays, alarms, and other controls.

## 4. Event-B and BMotion Studio

**Resources:**

- [*Validating the Requirements and Design of a Hemodialysis Machine Using iUML-B, BMotion Studio, and Co-Simulation*](https://doi.org/10.1007/978-3-319-33600-8_31)
- [*BMotionWeb: A Tool for Rapid Creation of Formal Prototypes*](https://doi.org/10.1007/978-3-319-41591-8_27)
- [BMotionWeb for ProB Handbook](https://stups.hhu-hosting.de/handbook/bmotion/current/html/index.html)

Another approach within the Event-B ecosystem uses BMotion Studio to create domain specific visualizations of formal models.
BMotion Studio uses ProB to animate the underlying Event-B model, but provides a more programmable environment for constructing an interactive interface around it.

The approach is demonstrated in a case study of a hemodialysis machine, a medical device that removes waste products from the blood of patients with kidney failure.
Unlike the automotive case study examined earlier, where Classical B is used during exploratory development before the model is translated to Event-B, the hemodialysis model is developed directly in Event-B using iUML-B (graphical event B editor) state machine and class diagrams.
The device is divided into a "top level" controller and a "low level" controller.
The top level controller manages the overall hemodialysis process and interactions with the user, while the low level controller manages individual machine operations and interacts more directly with the physical equipment.

The main process is modelled as a state machine containing the phases `PREPARATION`, `INITIATION`, and `ENDING`, together with a `STANDBY` state.
These phases are progressively refined using nested state machines that introduce more detailed sequences of behaviour.
For example, the preparation phase is refined into steps for testing control functions, connecting concentrate, setting parameters, preparing the machine, rinsing the dialyzer, and eventually connecting the patient.

The iUML-B state machines are translated into Event-B variables and events.
For example, the variable `CS_TopLevel` represents the current state of the top level controller.
Moving from `STANDBY` to `PREPARATION` is represented by the following Event-B event:

```event-b
HDSystem_Prepares:
when
    CS_TopLevel = STANDBY
then
    CS_TopLevel := PREPARATION
end
```

At the next refinement level, additional state is introduced to represent the more detailed behaviour within each phase.
For example, starting the control function testing process changes both the top level state and the state of the nested preparation process:

```event-b
HDSystem_StartsTestingCF:
when
    CS_TopLevel = STANDBY
then
    CS_TopLevel := PREPARATION
    Preparation_sm := CF_TESTING
end
```

The formal state is therefore represented by variables such as `CS_TopLevel` and `Preparation_sm`, while Event-B events describe how those variables can change.
Later refinements introduce additional variables representing lower level controller behaviour and physical quantities such as blood flow and pressure.

Separate state machines model lower level behaviours such as testing control functions, monitoring pressures, controlling blood flow, administering boluses, and detecting abnormal conditions.
The model also distinguishes between the controller and the physical equipment.

ProB is used to animate and model check the Event-B model, while Rodin is used to prove invariant properties.
BMotion Studio then provides a domain specific visualization on top of the ProB animation.

BMotion Studio divides the visualization into two main views: a representation of the machine's user interface and a representation of the physical environment of the hemodialysis machine.

The user interface view displays information that would normally be presented to an operator of the machine.
Dialysis parameters such as blood flow and pressure values are represented graphically, together with their current values and operating limits.
The interface also contains controls such as the machine's power button and indicators such as the automated self test signal lamp.

The connection between the formal state and the visualization can be seen through the blood flow display.
The current blood flow is represented by the Event-B variable `bloodFlow`.
BMotion Studio connects this variable to a graphical element using a formula observer:

```javascript
bms.observe("formula", {
    selector: "#bloodFlow",
    formulas: ["bloodFlow"],
    trigger: function(e, v) {
        e.text(v[0]);
    }
});
```

The selector `#bloodFlow` identifies the graphical element used to display the value.
Whenever the formal state changes, the observer evaluates `bloodFlow` and passes its current value to the trigger function, which updates the text displayed by the graphical element.
The graphical interface directly displays the value from the current Event-B state.

Interaction can also flow in the opposite direction through event handlers.
For example, the graphical power button is connected to the Event-B events `User_PressesOn` and `User_PressesOff`:

```javascript
bms.executeEvent({
    selector: "#bt_power",
    events: [
        {name: "User_PressesOn"},
        {name: "User_PressesOff"}
    ]
});
```

The selector `#bt_power` identifies the graphical power button.
Interacting with this control executes the corresponding event in the Event-B model through ProB rather than changing the visualization independently.
After the event executes, observers receive the resulting formal state and update the graphical elements accordingly.

![](assets/image17.png)
*Visualisation of UI display panel. Source: Validating the Requirements and Design of a Hemodialysis Machine Using iUML-B, BMotion Studio, and Co-Simulation*

The connection can therefore be summarized as follows:

| Event-B element | Role | BMotion Studio representation |
|---|---|---|
| `bloodFlow` | Current blood flow value in the formal state | Text displayed by the `#bloodFlow` graphical element |
| `User_PressesOn` | Event representing the user switching the machine on | Executed through the graphical power button |
| `User_PressesOff` | Event representing the user switching the machine off | Executed through the graphical power button |
| Resulting Event B state | New formal state after an event | Observers update the corresponding graphical elements |

The second view visualizes the physical environment of the hemodialysis machine.
It shows components such as the patient, dialyzer, blood pump, saline and heparin supplies, and arterial, venous, and blood entry pressure monitors.
Values produced by the formal model are displayed directly on the corresponding components, allowing the state to be interpreted in the context of the physical dialysis process rather than as a collection of abstract formal variables.

![](assets/image16.png)
*Visualisation of the environment of the HD machine. Source: Validating the Requirements and Design of a Hemodialysis Machine Using iUML-B, BMotion Studio, and Co-Simulation*

BMotion Studio therefore establishes a two-way connection between the formal model and a domain specific interface.
Observers translate changes in formal state into changes in graphical elements, while event handlers translate interactions with graphical elements into Event-B events executed through ProB.

This is similar to the architecture used by VisB, but BMotion Studio provides a more programmable mapping mechanism.
VisB primarily uses a JSON glue file to associate formal expressions with SVG attributes, whereas BMotion Studio uses JavaScript observers and event handlers that can perform more complex updates when the formal state changes.

The case study also demonstrates how this visualization fits into the wider development process.
The iUML-B diagrams provide a structural view of the sequential control logic, ProB animation and model checking expose possible behaviours and counterexamples, Rodin is used to prove suitable safety properties, and BMotion Studio presents the running model using a representation closer to the actual medical device.

This domain specific representation is particularly valuable as the model becomes more complex.
Instead of requiring a domain expert to interpret Event-B variables and enabled events directly, the state can be examined through familiar concepts such as pumps, pressure monitors, alarms, displays, and patient connections.
The visualization can therefore be used not only to inspect the formal model, but also to validate whether its behaviour corresponds to the system that the model is intended to represent.

## 5. TLA+ and Spectacle

**Resources:**

- [Spectacle](https://github.com/will62794/spectacle)
- [Spectacle animation documentation](https://github.com/will62794/spectacle/blob/master/doc/animation.md)
- [Spectacle lock server example](https://github.com/will62794/spectacle/blob/master/specs/lockserver.tla)
- [Spectacle lock server animation](https://github.com/will62794/spectacle/blob/master/specs/lockserver_anim.tla)
- [TLA+ documentation](https://lamport.azurewebsites.net/tla/tla.html)

Another approach to visualizing formal specifications is Spectacle, a browser based environment for exploring, visualizing, and sharing specifications written in TLA+.

TLA+ is a formal specification language designed primarily for modelling concurrent and distributed systems.
A system is represented using state variables, an initial state predicate describing the possible starting states, and actions describing how the variables may change between states.
A `Next` relation typically combines the actions that are allowed to occur and defines the possible transitions of the specification.
The TLC model checker can then explore reachable states, check invariants and deadlocks, and verify safety and liveness properties.

Spectacle provides a more interactive way to explore these specifications.
It allows states and transitions to be explored in the browser and traces to be viewed and shared.
A visualization can also be associated with a specification so that each state in a trace is represented using a domain specific graphical view rather than only through the values of its variables.

The connection between formal state and visualization can be seen clearly in Spectacle's lock server example.
The system contains a set of servers and clients, where each server manages a lock that can be acquired by a client.
Its formal state is represented primarily by two variables:

```tla
VARIABLE semaphore
VARIABLE clientlocks
```

The variable `semaphore` records whether the lock belonging to each server is available, while `clientlocks` records the set of server locks currently held by each client.

State transitions are represented using TLA+ actions.
For example, the `Connect` action allows a client to acquire a server's lock when it is currently available:

```tla
Connect(c, s) ==
    /\ semaphore[s] = TRUE
    /\ clientlocks' =
        [clientlocks EXCEPT ![c] = clientlocks[c] \cup {s}]
    /\ semaphore' =
        [semaphore EXCEPT ![s] = FALSE]
```

The unprimed variables describe the current state and the primed variables describe the resulting next state.
When `Connect(c, s)` occurs, the selected server becomes unavailable in `semaphore'` and the server is added to the client's set of held locks in `clientlocks'`.

The visualization is defined separately in an animation module that extends the original specification.
This gives the visualization access to the same state variables without placing graphical definitions directly in the formal model.

The animation module constructs SVG elements using TLA+ expressions.
For example, the servers are represented as circles whose fill colour depends directly on the corresponding value of `semaphore`:

```tla
cs ==
    [i \in ServerIdDomain |->
        Circle(
            20 * i,
            10,
            3,
            [
                fill |->
                    IF semaphore[ServerId[i]]
                    THEN "green"
                    ELSE "orange"
            ]
        )
    ]
```

Each server is therefore generated from the formal model rather than being manually connected to a preexisting SVG element.
If `semaphore[ServerId[i]]` is `TRUE`, the corresponding circle is green.
If the lock has been acquired and the value becomes `FALSE`, the circle becomes orange.

The final visualization is returned through the `AnimView` operator:

```tla
AnimView ==
    Group(
        cs,
        ("transform" :> "scale(2.5) translate(30 20)")
    )
```

When the selected formal state changes, Spectacle reevaluates `AnimView` using the values of the variables in that state and constructs the corresponding SVG representation.

The overall architecture keeps the original TLA+ specification separate from the visualization definitions.
The animation module extends the original specification to access its state variables and the SVG module to construct graphical elements.
Spectacle evaluates `AnimView` for the currently selected state in the browser and displays the resulting SVG alongside the trace explorer.


![](./assets/tla.png)
*Architecture of the Spectacle visualization approach, showing how a separate animation module extends the original TLA+ specification and uses the SVG module to define `AnimView`.*

The resulting interface combines the trace explorer with the SVG representation generated for the selected state.
![](assets/spectacle1.png)

![](assets/spectacle2.png)
*Spectacle displaying the lock server trace and its corresponding SVG visualization.*

The connection between individual parts of the formal model and the visualization can therefore be summarized as follows:

| TLA+ element | Role | Spectacle representation |
|---|---|---|
| `semaphore` | Records whether each server lock is available | Determines the colour of each server circle |
| `clientlocks` | Records the locks held by each client | Available to the visualization for representing client ownership |
| `Connect(c, s)` | Transition that allows a client to acquire a lock | Produces a new state that changes the visualization |
| `AnimView` | Function of the current formal state | Produces the SVG representation displayed by Spectacle |

This differs from the mapping approaches used by VisB and BMotion Studio.
VisB begins with an external SVG and uses a JSON glue file to map formal expressions onto individual SVG attributes, while BMotion Studio uses observers to update existing graphical elements.
Spectacle instead constructs the visualization programmatically from the current formal state.

This approach naturally supports collections of model objects.
For example, the lock server animation constructs a collection of circles from the set of servers rather than requiring every server to be manually associated with an individual SVG element.
If the structure of the model changes, the visualization can be regenerated from the corresponding collection.

Spectacle is also closely connected to trace exploration.
Because a TLA+ specification may permit several possible next states, a user can explore different behaviours of the model and view the visualization associated with each state along a selected trace.
This makes the visualization useful for understanding nondeterministic and concurrent behaviour, where the same state may have several possible successors.

The visualization remains separate from the original specification.
The underlying TLA+ model can still be checked using TLC without requiring the animation definitions.
The animation module extends the original specification only to read its state and construct a graphical representation of it.

Spectacle therefore demonstrates a visualization approach in which graphical output is defined programmatically as a function of formal state.
Rather than maintaining a separate graphical object to variable mapping, the visualization code reads the specification variables directly and constructs the appropriate SVG representation for each state.
Its integration with trace exploration also makes it particularly useful for visualizing sequences of states and alternative behaviours in concurrent or nondeterministic systems.

## 6. Electrum and the Electrum Analyzer

**Resources:**

- [*Validating Multiple Variants of an Automotive Light System with Electrum*](https://doi.org/10.1007/978-3-030-48077-6_26)
- [*The Electrum Analyzer: Model Checking Relational First-Order Temporal Specifications*](https://doi.org/10.1145/3238147.3240475)
- [Electrum](https://haslab.github.io/Electrum/)
- [Electrum automotive lighting case study](https://github.com/haslab/Electrum2/wiki/ELS)

The automotive lighting case study examined earlier with B and Event-B has also been modelled using Electrum.
This provides another direct comparison with the Spectabular implementation, but with a substantially different approach to modelling, verification, and visualization.

Electrum is a formal specification language that extends Alloy with mutable relations and temporal logic.
Alloy models structure using signatures, which represent sets of objects, and fields, which represent relations between those objects.
Electrum allows signatures and relations to be declared as variable, so their values can change between states and describe the behaviour of a system over time.

The automotive model follows the signal based architecture of the original case study.
Inputs from the driver and environment are represented as signals, while outputs represent actuators such as the vehicle lights.
Signals are organized using signature hierarchies, and their current values are represented using mutable relations.

For example, the low beam lights are represented by a hierarchy of signatures with a mutable `state` field:

```electrum
abstract sig Light extends Signal {
    var state : one LightState
}

abstract sig LowBeam extends Light {}

one sig LowBeamLeft, LowBeamRight extends LowBeam {}
```

Possible light states are represented by another hierarchy:

```electrum
abstract sig LightState extends State {}

abstract sig Full, Off extends LightState {}

one sig Half, Low extends LightState {}

one sig On, Temp extends Full {}
```

The formal state of a light is therefore represented by the value of its `state` relation.
For example, `LowBeamLeft.state` gives the current state of the left low beam, while `LowBeam.state` refers collectively to the states of both low beams.

Boolean signals use an even simpler representation.
A mutable set called `SignalOn` contains all Boolean signals that are currently active:

```electrum
var sig SignalOn in BooleanSignal {}
```

A Boolean signal is therefore active when it belongs to `SignalOn`.

System behaviour is expressed using relational and temporal constraints over the current and next states.
Primed expressions refer to the value of a relation in the succeeding state.

For example, one requirement states that when the ignition is on and the light rotary switch is set to on, the low beams should be activated:

```electrum
KeyState.state in KeyInIgnitionOnPosition and
LightRotarySwitch.state in LSOn
implies
LowBeam.state' in On
```

Here, `LowBeam.state` represents the current light state, while `LowBeam.state'` represents its value in the next state.
Unlike Event-B, where an explicitly named event changes the state, Electrum describes the relationship that must hold between successive states.

Real time is abstracted rather than represented using exact durations.
For example, the temporary state `Temp` is used to represent that the low beams are still within a timed activation period without requiring a specific number of states to correspond to three seconds.
A liveness constraint ensures that this temporary state is eventually left:

```electrum
low in Temp implies eventually low not in Temp
```

This allows timed behaviour to be represented at a higher level of abstraction, but exact timing information such as a one second blinking period or a three second delay cannot be verified directly in this model.

The Electrum Analyzer can generate executions satisfying the model and display each state graphically.
Unlike VisB, PVSio-web, and BMotion Studio, the visualization does not begin with an externally supplied domain specific image.
Instead, the Analyzer automatically constructs a graph from the signatures, atoms, fields, and relations that exist in the generated formal state.

![](./assets/image27.png)
*An example Analyzer visualization of the model above. Source: Validating Multiple Variants of an Automotive Light System with Electrum*

The appearance of this graph can be customized using a theme.
Colours, shapes, labels, and visibility can be changed, and relations can be displayed either as edges or as attributes.
The model can also contain auxiliary elements used only to improve the visualization.

For example, the automotive model introduces structural elements representing parts of the vehicle:

```electrum
one sig Car, LeftSide, RightSide, Menu, UCP {}
```

Auxiliary relations then associate formal signals with these graphical groupings:

```electrum
fun _actuators : univ -> univ {
    LeftSide ->
        (BlinkLeft + LowBeamLeft + TailLampLeft + ...)
    +
    RightSide ->
        (BlinkRight + LowBeamRight + TailLampRight + ...)
}
```

Another auxiliary function groups signals that are currently active:

```electrum
fun _on : set univ {
    state.Full + state.(LSOn + LSAuto) + SignalOn + ...
}
```

The visualization theme can then give the elements returned by `_on` a distinguishing appearance, such as a different colour.
The graphical representation is therefore derived from the same signatures and relations used by the formal model, with additional helper relations and theme settings used to make the resulting graph easier to interpret.

The connection can be summarized as follows:

| Electrum element | Role | Analyzer representation |
|---|---|---|
| `LowBeamLeft`, `LowBeamRight` | Formal objects representing the low beam actuators | Nodes in the generated instance graph |
| `state` | Mutable relation containing the current value of a signal | Relation or attribute displayed for each signal |
| `SignalOn` | Set of currently active Boolean signals | Used to determine which signals are active |
| `_actuators` | Auxiliary visualization relation | Groups signals according to their position in the vehicle |
| `_on` | Auxiliary set of active elements | Allows the theme to visually distinguish active signals |
| Resulting Electrum state | Assignment of atoms and relations at one point in a trace | Automatically generated graph displayed by the Analyzer |

This is different from the visualization mappings used by VisB and BMotion Studio.
Those tools begin with a supplied graphical representation and explicitly connect formal state to graphical elements.
In Electrum, the Analyzer instead derives the visualization directly from the relational structure of the formal state.
Themes and auxiliary relations can improve this representation, but the result remains primarily a graph of the model rather than an arbitrary domain specific visualization of the physical system.

The Analyzer is also closely integrated with scenario and trace exploration.
Scenarios are expressed as constraints over sequences of states and executed using `run` commands.
For example, a simple expected low beam behaviour can be described as:

```electrum
pred LowBeam2Exp {
    LowBeam.state in OffP;
    always LowBeam.state in Half
}
```

The Analyzer searches for a trace satisfying these constraints and allows the user to move through its states.
Because several traces may satisfy the same constraints, the user can also request alternative configurations, initial states, or transitions.

This makes Electrum particularly useful for exploring nondeterministic behaviour.
Rather than requiring one specific execution to be chosen in advance, the Analyzer can generate alternative valid behaviours and display the formal state associated with each step.

Requirements are verified separately using temporal assertions.
For example, the expected low beam behaviour can also be expressed as a property that must hold for all executions:

```electrum
assert ELS14 {
    always (
        KeyState.state in KeyInIgnitionOnPosition and
        LightRotarySwitch.state in LSOn
        implies
        LowBeam.state' in Full
    )
}
```

The Analyzer can check these assertions using bounded or unbounded model checking and presents counterexamples using the same trace and graph visualization used for generated scenarios.

Electrum therefore provides a particularly close connection between formal state, automated analysis, and visualization.
Its visualization requires little additional mapping because the Analyzer can display the relational structure of the model directly, and the same interface supports generated scenarios, alternative traces, and counterexamples.
However, this also limits how closely the visualization can resemble the physical system when compared with approaches based on arbitrary SVG graphics or realistic device interfaces.

## 7. Simulink and Stateflow

**Resources:**

- [Simulink](https://www.mathworks.com/products/simulink.html)
- [Stateflow](https://www.mathworks.com/products/stateflow.html)
- [Simulink Dashboard](https://www.mathworks.com/help/simulink/dashboard-blocks.html)
- [Simulate Asynchronous Services for Vehicle Headlight Management](https://www.mathworks.com/help/systemcomposer/ug/simulate-asynchronous-interfaces-for-headlights.html)
- [Simulation Data Inspector](https://www.mathworks.com/help/simulink/slref/simulationdatainspector.html)
- [Sequence Viewer](https://www.mathworks.com/help/systemcomposer/ref/sequenceviewer-app.html)

Simulink and Stateflow provide another style of visualization based around graphical simulation models.
Simulink represents systems using interconnected blocks and signals, while Stateflow represents state based behaviour using graphical state machines.

A MathWorks example demonstrates vehicle headlight management using asynchronous services in a System Composer software architecture model.
The architecture model, `HeadlightArch`, contains a lighting manager component, `LightingManager`, separate `LeftHeadlight` and `RightHeadlight` components, and a `Logging` component.

The left and right headlights are separate instances of the same referenced Simulink model, `HeadLight`.
This allows both headlights to reuse the same behaviour while maintaining separate component state.

The referenced headlight model provides two Simulink functions:

```text
setMode(lightMode)
getMode()
```

The `setMode` function changes the current lighting mode and returns a value indicating whether the headlight is broken.
The `getMode` function returns the current lighting mode.

A single service interface connects the lighting manager to both headlight instances.
The manager can therefore call `setMode` or `getMode` for either the left or right headlight through the same interface.

The connection can be summarized as follows:

| Simulink element | Role |
|---|---|
| `LightingManager` | Determines when the headlight mode should be changed or inspected |
| `LeftHeadlight` | Left instance of the reusable `HeadLight` model |
| `RightHeadlight` | Right instance of the reusable `HeadLight` model |
| `setMode(lightMode)` | Changes the operating mode of a selected headlight |
| `getMode()` | Retrieves the current mode of a selected headlight |
| `Logging` | Records function call and status information during simulation |

The service functions are configured to execute asynchronously.
When the lighting manager requests a function call, the headlight component processes the request according to the priorities configured in the Functions Editor rather than only according to the order in which requests arrive.

If a higher priority function is called while a lower priority function is running, the higher priority function can execute before the lower priority function finishes.
If a lower priority function is called while a higher priority function is running, it waits until the higher priority function completes.

The `LightingManager` component references a Simulink behaviour model containing two function call subsystems.
The `changeLightMode` subsystem calls `setMode` for the left and right headlights, while the `checkLights` subsystem calls `getMode` to retrieve their current states.

Because these calls execute asynchronously, the Function Caller blocks produce message outputs.
These messages are processed by Message Triggered Subsystem blocks when the corresponding service calls return.

![](./assets/image22.png)

The architecture can be represented as follows:

![](./assets/simulink_uml.png)


During simulation, the Simulation Data Inspector displays the logged results of calls to `setMode` and `getMode` for both headlights.
These signals can be viewed as plots over simulation time, making it possible to inspect when the lighting modes change and when their values are retrieved.

![](./assets/image24.png)

The Sequence Viewer provides a different visualization of the same execution.
It displays the interactions between the lighting manager and the headlight components, including the order in which asynchronous function calls and returned messages are processed.
Changing the priorities in the Functions Editor changes the resulting execution order shown in this view.

![](./assets/image26.png)

The example therefore provides several related visual representations of the model.
The System Composer diagram represents the software architecture, the referenced Simulink models represent the component behaviour, the Simulation Data Inspector displays signal values over time, and the Sequence Viewer displays communication between the components.

A domain oriented interface could be added using Simulink Dashboard components.
Dashboard controls could be connected to input signals or parameters used by `LightingManager` to request new light modes, while lamps or displays could be connected to the resulting left- and right headlight mode and status signals.
A simplified example controlling the light mode is shown below:

![](./assets/image23.png)

The full extended visualization architecture would therefore be:

![](./assets/simulink_uml1.png)

This Dashboard extension would provide a two-way interaction similar to the device interfaces used by PVSio-web.
A user could modify an input through a graphical control, the Simulink model would execute the corresponding service calls, and the resulting signals would update the graphical displays.

However, the Dashboard does not replace the other Simulink visualizations.
It presents the system through controls and indicators, while the System Composer model exposes component structure, the Simulation Data Inspector presents signal histories, and the Sequence Viewer presents asynchronous communication.

The Simulink ecosystem also provides extensive support for simulation, testing, verification, and code generation.
Simulation time and asynchronous execution are part of the model itself rather than being introduced only by the visualization.

Its standard visualizations are focused primarily on software architecture, engineering diagrams, controls, signal plots, and communication traces.
A richer domain specific visualization, such as a graphical vehicle whose headlights illuminate and animate, would require a custom interface or additional visualization logic.

Simulink also executes a model using supplied inputs and simulation settings rather than exposing alternative solver generated next states in the same way as ProB, Spectacle, or Electrum.
Its replay and inspection facilities are strong for recorded simulations, but are not primarily designed for interactively exploring nondeterministic formal transitions.

This approach is therefore useful as an example of integrating component modelling, asynchronous execution, simulation, visualization, and testing within the same industrial environment.
It also demonstrates the value of reusable component models, service interfaces, Dashboard controls, signal plots, and sequence diagrams.

A Spectabular visualization system should provide benefits beyond reproducing these existing capabilities.
Its distinguishing features could include a direct connection to formal tabular specifications, arbitrary SVG based domain representations, exploration of alternative formal transitions, and the use of the same visualization for interactive execution, scripted scenarios, and counterexample replay.

## 8. Other Formal Approaches with Non Visual Animations

Several other formal approaches provide simulation, trace exploration, plotting, or graphical modelling without providing a domain specific animated visualization comparable to VisB, BMotion Studio, PVSio-web, or Spectacle.
These approaches are summarized briefly below.

### Event-B Mechanical Lung Ventilator

**Resource:**

- [*An Event-B Model of a Mechanical Lung Ventilator*](https://doi.org/10.1007/978-3-031-63790-2_25)

The ABZ 2024 Mechanical Lung Ventilator case study has been modelled in Event-B using refinement to progressively introduce the behaviour of the ventilator.
The model is developed and proved using Rodin and validated using the ProB animator and model checker.
ProB allows the model's events and execution traces to be explored, but this work does not introduce a dedicated domain specific visualization of the ventilator.
It therefore provides animation at the level of the formal model rather than a graphical representation of the physical system.

### FRET and Event-B

**Resource:**

- [*FRETting and Formal Modelling: A Mechanical Lung Ventilator*](https://doi.org/10.1007/978-3-031-63790-2_28)

Another approach to the Mechanical Lung Ventilator combines NASA's Formal Requirements Elicitation Tool (FRET) with Event-B.
FRET is used to formalize requirements, which then guide the construction of an Event-B model for verification.
The workflow supports reasoning about system behaviour and evaluating requirements against execution traces, but does not provide a domain specific animated representation of the ventilator.
Its visualization is therefore focused more on requirements and traces than on visually representing the running system.

### mCRL2

**Resource:**

- [*Modelling and Analysing a Mechanical Lung Ventilator in mCRL2*](https://doi.org/10.1007/978-3-031-63790-2_27)

The Mechanical Lung Ventilator has also been modelled using the process algebra mCRL2.
Its functional requirements are expressed formally and checked using model checking, while the mCRL2 toolset allows the behaviour of the specification to be simulated and explored through execution traces.
This provides useful support for examining possible system behaviours and counterexamples, but the simulation is trace oriented rather than a domain specific visualization of the ventilator.

### TASTD and cASTD

**Resource:**

- [*Modelling a Mechanical Lung Ventilation System Using TASTD*](https://doi.org/10.1007/978-3-031-63790-2_26)

Timed Algebraic State-Transition Diagrams (TASTD) provide a graphical notation for modelling the Mechanical Lung Ventilator, including its sensors, actuators, and timing constraints.
The cASTD compiler translates the specification into executable C++ code, which can then be run in simulation mode against the test sequences provided with the case study.
The ASTD diagrams provide a useful graphical representation of the structure and control flow of the specification, but they are not themselves animated as the simulation executes.
The visualization is therefore mainly used to describe the model rather than to provide a domain specific view of its runtime behaviour.

![](./assets/astd2.webp)

*Example ASTD diagram from the Mechanical Lung Ventilator. Source:Modelling a Mechanical Lung Ventilation System Using TASTD* 

### Real-Time CCSL and MRTCCSL

**Resources:**

- [*Real-Time CCSL: Application to the Mechanical Lung Ventilator*](https://doi.org/10.1007/978-3-031-63790-2_24)
- [MRTCCSL](https://github.com/PaulRaUnite/mrtccsl)

The Mechanical Lung Ventilator has also been specified using the Clock Constraint Specification Language (CCSL), which represents causal and temporal behaviour using constraints between logical clocks.
The MRTCCSL tool can simulate these clock constraints over an execution trace and produce graphical plots showing their behaviour.
These graphs are useful for understanding timing relationships and validating a trace, but they visualize temporal data rather than presenting a domain specific animated representation of the ventilator.

![](./assets/astd2.webp)

*Example MRTCCSL diagram from the Mechanical Lung Ventilator. Source:Real-Time CCSL: Application to the Mechanical Lung Ventilator* 

### KeYmaera X

KeYmaera X is a verification tool for hybrid systems that combine discrete control with continuous physical behaviour.
It provides plotting facilities that can help visualize trajectories and the evolution of system values.
These plots are useful for analysing continuous behaviour, but they do not provide the kind of customizable domain specific animation considered here.

### Ptolemy II

Ptolemy II uses graphical models composed of actors that communicate through ports.
The graphical representation is an important part of constructing and understanding the model, and different models of computation can be used to describe how the actors execute and interact.
However, the visualization primarily represents the structure of the computational model itself rather than producing a separate domain specific visualization of the system while it executes.

![](./assets/image25.png)
*An example Ptolemy II based visualization. Source: Claudius Ptolemaeus, Editor, System Design, Modeling, and Simulation using Ptolemy II, Ptolemy.org, 2014. http://ptolemy.org/books/Systems.* 

### CSP, RoboChart, and PAT

**Resources:**

- [RoboChart Reference Manual](https://robostar.cs.york.ac.uk/publications/techreports/reports/robochart-reference.pdf)
- [RoboTool](https://robostar.cs.york.ac.uk/robotool/)
- [Process Analysis Toolkit](https://pat.comp.nus.edu.sg/)
- [PAT Simulator](https://pat.comp.nus.edu.sg/resources/OnlineHelp/htm/scr/2%20Getting%20Started/2.2.2%20Simulator%20.htm)

CSP and related process algebra tools provide mechanisms for exploring processes, traces, and possible interactions between concurrent components.
RoboChart and the Process Analysis Toolkit provide two relevant forms of graphical support, although neither provides the same kind of domain specific visualization as VisB or BMotion Studio.

RoboChart is a graphical state machine notation designed for modelling robotic controllers.
A RoboChart model represents the robotic platform, controllers, state machines, events, operations, and timed transitions using architectural and state machine diagrams.
RoboTool validates these graphical models and automatically generates their formal CSP semantics, which can then be analysed using model checking.

The graphical diagrams make the structure and control flow of a CSP based model easier to understand.
However, they primarily visualize the specification itself rather than presenting a separate graphical representation of the robot as its state changes.
RoboChart simulations can be connected to robot simulation environments, but the resulting physical visualization is provided by the target simulator rather than by a general mapping between CSP state and arbitrary graphical objects.

![](./assets/image28.png)
*An example RoboChart diagram for a chemical detector. Source: Pg 144, RoboChart Reference Manual* 

PAT provides more direct support for visualizing the execution of CSP models.
Its CSP# language retains the main CSP process operators while adding variables, arrays, control structures, and shared memory or message passing communication.

The PAT simulator displays the enabled events, the values of variables in the selected state, the executed event trace, and a graph of the visited states and transitions.
The user can choose an enabled event, perform a random simulation, generate a bounded state graph, move back to an earlier state, or automatically replay a trace.
Event sequences can also be entered as scripts, allowing a particular scenario to be reproduced.

Counterexamples produced by the model checker can be opened in the same simulator and replayed step by step.
PAT therefore connects verification results directly to trace and state visualization rather than requiring the counterexample to be interpreted only as text.

PAT includes a small number of examples with specialized graphical views, such as board representations for the dining philosophers and sliding game models.
These are example specific implementation rather than a general library through which users can bind arbitrary graphical elements to CSP events and variables.

![](./assets/patsim.JPG)
*The PAT Simulator window for the sliding game. [Source](https://pat.comp.nus.edu.sg/resources/OnlineHelp/htm/index.htm#page=2%20Getting%20Started/2.2.2%20Simulator%20.htm)* 


RoboChart provides graphical architectural and state machine modelling, while PAT provides interactive state graphs, variable inspection, trace playback, and counterexample visualization.
Their visualizations remain focused on the structure and execution of the formal model rather than on a customizable domain specific representation of the system.
No directly comparable general purpose mechanism for mapping CSP state and events onto an arbitrary SVG or device interface was identified.

### BRAMA

BRAMA was also encountered as a tool for creating graphical interfaces around formal models and appears similar in purpose to BMotion Studio.
However, the tool is closed source and was not sufficiently accessible to evaluate in the same detail as the other approaches.
It is therefore not considered further in the comparison.

## 9. Case Studies to Support

The following case studies cover a range of behaviours that would be useful for evaluating visualization support in Spectabular.
Together, they include moving objects, spatial environments, interactive device interfaces, concurrent entities, timed behaviour, data plots, and replayable execution scenarios.

### Safety Controller for Autonomous Driving — ABZ 2025

[*Safety Controller for Autonomous Driving*](https://abz-conf.org/site/2025/casestudy/) models a safety controller for autonomous vehicles travelling on a highway.
The simpler version considers vehicles accelerating and braking on a single lane while maintaining a safe distance, while the extended version introduces multiple lanes and lane changes.
A visualization could show the vehicles moving along a road while their relative positions and safety distances change as the model executes.

![enter image description here](assets/image13.png)

This case study would therefore test support for moving graphical objects, multiple interacting entities, and the visualization or replay of continuous movement.

### Planetary Rover — ABZ 2026

The [*Planetary Rover*](https://abz-conf.org/site/2026/casestudy/) case study concerns the behaviour of an autonomous planetary rover.
A useful visualization could display a map of the operating environment, important locations or mission points, the rover's current position, and the path it has travelled.
This would test the ability to visualize movement through a larger map environment and to dynamically update both an object's position and its history or planned route.

### Mechanical Lung Ventilator — ABZ 2024

The [*Mechanical Lung Ventilator*](https://abz-conf.org/site/2024/casestudy/) provides ventilation support in two main operating modes: Pressure Controlled Ventilation, where ventilation is controlled by the machine, and Pressure Support Ventilation, where the machine assists breathing initiated by the patient.
The case study contains significant timed behaviour and also includes the ventilator's user interface as part of the system being considered.
A visualization could represent the ventilator itself together with its controls, alarms, operating mode, sensor readings, and changing respiratory values.
This would test support for interactive device interfaces, gauges and displays, alarms, and timed behaviour.

### Hemodialysis Machine — ABZ 2016

The [*Hemodialysis Machine*](https://abz-conf.org/case study/abz16/) case study describes the control and safety requirements of a machine used to perform hemodialysis.
As seen in the BMotion Studio example, the system contains a rich physical environment involving the patient, dialyzer, pumps, pressure monitors, supplies, and a user interface.
Supporting this case study would test the ability to combine a domain specific diagram with interactive controls, numerical readings, alarms, and changing states of physical components.

### Hybrid ERTMS/ETCS Level 3 — ABZ 2018

The [*Hybrid ERTMS/ETCS Level 3*](https://abz-conf.org/case study/abz18/) case study concerns railway signalling and train movement management using virtual subsections of track.
The system must accommodate several types of trains while safely managing their positions and occupancy of the railway sections.
A visualization could display the railway as a network of track sections and show the positions and movements of multiple trains as the formal state changes.
This would test support for multiple moving objects at the same time, shared resources, and visualization of safety properties such as track occupancy.

### AMAN Arrival Manager — ABZ 2023

The [*AMAN Arrival Manager*](https://abz-conf.org/case study/abz23/) is a partly autonomous system for scheduling the landing sequence of aircraft at an airport.
It combines automated scheduling with interactions from Air Traffic Controllers, who can modify aspects of the proposed arrival schedule.
A visualization could show aircraft approaching an airport together with their ordering, scheduled arrival information, and changes made by either the automated system or the controller.
This would test support for multiple concurrent entities, interactive modification of a running model (at a large scale), and visualization scheduling information.

### Pacemaker

The [*Pacemaker System Specification*](https://greg4cr.github.io/courses/fall17csce740/Documents/PACEMAKER.pdf) describes an implanted pacemaker together with its Device Controller-Monitor and leads.
The system includes configurable pacing modes, sensing and pacing behaviour, diagnostic information, historical measurements, and real-time electrograms.
A visualization could combine a representation of the pacemaker and heart with a user interface showing the internal pacing and sensing state, diagnostic values, and real-time signal plots.
This case study would therefore test whether a visualization can combine domain graphics with numerical, textual, and time series information rather than representing the system using only a single animated scene.

### Elevator

The [elevator example](https://ieeexplore.ieee.org/document/9988440) provides a smaller system with naturally sequenced behaviour involving requests, movement between floors, arrival, and changes to the state of the elevator and its controls.
A visualization could use prebuilt graphical assets to represent the elevator, floors, doors, call buttons, and current requests while showing the execution of these events in sequence.

![](assets/image15.png)

This provides a useful case for supporting rich diagrams and replayable scenarios without requiring the complexity of the larger automotive, railway, or medical case studies.

## 10. Comparison of Visualization and Animation Approaches

The approaches examined above differ mainly in how the visualization is supplied, how it is connected to the formal model, how execution is controlled, and whether the visualization supports timing, movement, replay, and domain specific interaction.

| Approach | Viz Source | Model Connection | Execution | Replay | Best Fit |
|---|---|---|---|---|---|
| VisB | External SVG | JSON glue | ProB operations | ProB traces | General SVG |
| PVSio-web | Device image | Region binding | Device interaction | Limited | Device interfaces |
| BMotion Studio | External SVG | JS observers | ProB operations | ProB traces | Multi view |
| Spectacle | Programmatic | TLA+ function | Trace stepping | Trace export | Concurrent systems |
| Electrum | Auto generated | Relational | Scenario search | Counterexamples | Structural analysis |
| Simulink | Block diagrams | Signal wiring | Simulation | Data Inspector | Engineering systems |


The following comparison uses the same categories for each approach:

- how the visualization is created and connected to model state;
- how the execution and animation are controlled;
- how timing, movement, and replay are handled;
- which kinds of case studies the approach supports well.

### VisB

#### Visualization source and connection to the model

VisB provides a relatively lightweight way to add domain specific visualization to an existing B or Event-B model.
The user creates an SVG in an external editor and writes a JSON glue file that maps SVG object identifiers to expressions over the formal state.

ProB remains entirely responsible for the model state and enabled operations.
VisB asks ProB to evaluate expressions over the current state and applies the results to SVG attributes such as fill, visibility, opacity, text, or position.
The glue file can also associate SVG objects with model operations, allowing interactions with the visualization to change the formal state.

#### Control, timing, and movement

Execution is controlled interactively through enabled ProB operations or through graphical elements connected to those operations.
The visualization changes when a new formal state is produced.

Timing is handled by the formal model, usually through explicit time passing events and timer variables.
VisB can display timed state changes, but does not provide its own independent animation timeline.

The visualization is primarily based on discrete changes to SVG attributes.
Objects can be moved by changing attributes such as position or transforms, but more complex or continuous motion must be done separately.

#### Replay and verification

Nondeterminism, trace exploration, model checking, and replay are provided by ProB.
A trace can be stepped through while VisB displays the corresponding visualization for each state.
Counterexamples and erroneous states found by ProB can therefore also be inspected through the same graphical representation.

#### Case study fit

VisB supports arbitrary SVG graphics and fits case studies with identifiable graphical components, including the automotive lighting system, railway networks, and moving systems such as elevators.
It can also represent device interfaces, although creating controls and displays requires manually constructing the SVG and glue mappings.

Its main limitation is that complex scripted animation is not directly supported.
It is strongest when the case study can be understood through discrete changes to graphical objects.

### PVSio-web

#### Visualization source and connection to the model

PVSio-web is particularly well suited to systems where the visualization is an interactive device interface.
A designer begins with an image of the device and places interactive and display regions over it.

Interactive regions are connected to executable PVS functions, while display regions are connected to values returned from the PVS state.
When the user interacts with the image, PVSio executes the corresponding function and returns a new state that is used to update the interface.

#### Control, timing, and movement

Execution is controlled primarily through interaction with the graphical prototype.
Buttons and other regions invoke PVS functions, making the workflow resemble the use of the physical device.

The visualization is mainly based on discrete updates to buttons, displays, alarms, and other interface elements.
It is less suited to visualizations involving objects moving through a larger scene.

Timing can be represented when it is included in the executable PVS model, but the visualization is not centred on trace timing or a separate animation timeline.

#### Replay and verification

PVSio-web is less focused on nondeterministic exploration and trace replay.
Its evaluator executes interactions as a deterministic sequence, and scenarios and traces are not exported or replayed in the same way as ProB or Spectacle.

Verification remains within the wider PVS environment.
The same PVS model can be used for theorem proving, but PVSio-web itself is primarily intended for interactive prototyping and validation.

#### Case study fit

PVSio-web fits device interface case studies particularly well, including ventilators, pacemakers, and other medical systems with buttons, displays, alarms, and gauges.
It provides a relatively easy way to construct a realistic prototype that can be shown to domain experts or regulators.

It is less appropriate for the railway, elevator, or other moving entity case studies because its visualization model is centred on regions placed over a device image.

### BMotion Studio

#### Visualization source and connection to the model

BMotion Studio provides a more programmable variation of the same general architecture as VisB.
The user creates a domain specific SVG visualization and connects it to an Event-B model animated through ProB.

Observers evaluate Event-B predicates and expressions whenever the formal state changes and use the results to update graphical elements.
Event handlers connect graphical controls to Event-B events, allowing interactions with the visualization to execute transitions in the model.

#### Control, timing, and movement

Execution can be controlled through both ProB and the graphical interface.
The visualization can therefore be used to inspect the current state or to invoke enabled events.

Like VisB, BMotion Studio is based mainly on discrete changes to SVG properties.
Its JavaScript observers allow more complex update logic and make it possible to reuse behaviour across groups of elements, but continuous movement and transitions are not provided automatically.

Timing is controlled by the Event-B model.
The visualization reflects timed states and events when they are represented formally.

#### Replay and verification

Nondeterminism, trace exploration, model checking, and replay are inherited from ProB.
BMotion Studio can display states reached during animation or counterexample exploration using the same domain specific interface.

Its debugging support also allows variables and invariants to be inspected alongside the visualization.

#### Case study fit

BMotion Studio is well suited to case studies that benefit from several coordinated views of the same system.
The hemodialysis example demonstrates both a device interface and a representation of the physical environment.

It could also support automotive, railway, elevator, and other SVG based case studies.
As with VisB, more complex movement would need to be implemented through explicit changes to graphical properties.

### Spectacle

#### Visualization source and connection to the model

Spectacle defines the visualization programmatically as a function of the current TLA+ state.
Rather than beginning with an external SVG and mapping individual objects to variables, an `AnimView` expression constructs SVG elements directly from specification values.

When the selected state changes, Spectacle reevaluates `AnimView` and produces a new graphical representation.
This works particularly well when the visualization contains collections of objects whose number or state depends on the model.

#### Control, timing, and movement

Execution is controlled through the Spectacle trace explorer.
The user can step through a trace and inspect the visualization produced for each state.

Timing must be represented in the TLA+ specification.
The visualization can read time variables and show timed changes, but it does not introduce timing independently.

Because SVG objects are constructed programmatically, their positions and other properties can depend on the current state.
Moving objects are therefore possible, although smooth motion between states would require additional animation behaviour beyond simply reevaluating the view.

#### Replay and verification

Spectacle is closely integrated with trace exploration.
Traces can be stepped through, shared, exported, and replayed through the visualization.
Alternative behaviours produced by nondeterministic and concurrent TLA+ specifications can also be explored.

The visualization is less directly connected to automatic debugging information than ProB based tools.
Current variables and expressions can be inspected, but invariants are not presented automatically through the visualization.

#### Case study fit

Spectacle is particularly suitable for concurrent and nondeterministic case studies such as the lock server example.
Its programmatic construction of SVG objects also fits systems containing collections of vehicles, trains, clients, servers, or other repeated entities.

The automotive lighting, railway, and elevator case studies could be represented in principle.
Device interface examples are also possible, but constructing a detailed interface programmatically may require more work than placing controls over an existing image as in PVSio-web.

### Electrum Analyzer

#### Visualization source and connection to the model

The Electrum Analyzer focuses on visualizing the mathematical structure of the model rather than producing a separate realistic domain image.
It automatically displays generated instances and traces as graphs of atoms and relations.

Themes control colours, shapes, labels, hidden elements, and whether relations appear as edges or attributes.
Visual helper elements and relations can be added to the model to improve the layout and interpretation of the graph without affecting the analysis.

#### Control, timing, and movement

Execution is controlled through analysis commands and scenario exploration in the Analyzer.
Users can generate traces, inspect successive states, and request alternative configurations or transitions.

Timing is represented through temporal constraints in the Electrum model.
In the automotive case study, exact real-time is abstracted, so the visualization shows changes between states rather than the precise duration of those states.

The graph layout may change between states, but the Analyzer is not intended for domain specific movement or continuous animation.

#### Replay and verification

Electrum provides strong support for scenario animation, nondeterministic traces, and counterexample exploration.
Alternative satisfying traces can be generated, and complete traces can be inspected through the same graphical interface.

Verification is closely connected to the visualization because counterexamples produced by model checking are shown using the same graph and trace views as ordinary generated scenarios.

#### Case study fit

The Analyzer is useful for case studies where understanding objects, relations, configurations, and alternative behaviours is more important than presenting a realistic interface.
It works well for the automotive model as a structural and behavioural graph and could represent the internal structure of railway, elevator, or concurrent system case studies.

It is less suitable for medical device interfaces or realistic moving scenes because it does not provide arbitrary domain specific graphics.
Its visualization remains recognizably a graph of the formal model.

### Simulink and Stateflow

#### Visualization source and connection to the model

Simulink and Stateflow integrate visualization directly into the modelling and simulation environment.
The model itself is represented using block diagrams, component architectures, and state machine charts.

Dashboard interfaces are developed graphically using controls and displays such as switches, lamps, gauges, and scopes.
Controls connect to model parameters or input values, while displays connect directly to signals produced during simulation.

#### Control, timing, and movement

Simulation can be started, paused, and inspected through the modelling environment.
Dashboard controls allow the user to change model inputs, while Stateflow can highlight active states and transitions during execution.

Timing is part of the simulation model.
Signal values, asynchronous calls, and state machine transitions execute according to simulation time rather than through a separate visualization mechanism.

Standard dashboards are mainly intended for controls, indicators, and signal plots.
More complex moving objects or domain scenes require additional visualization logic, although other parts of the Simulink ecosystem can connect simulations to richer visual environments.

#### Replay and verification

Simulation signals can be logged and replayed using tools such as the Simulation Data Inspector.
The Sequence Viewer can display the order of interactions between components, including asynchronous service calls.

These tools provide strong support for inspecting recorded simulations.
However, Simulink does not normally expose alternative solver generated next states in the same way as ProB, Spectacle, or Electrum.
Formal verification also requires additional tools beyond the standard simulation workflow.

#### Case study fit

Simulink and Stateflow are particularly suitable for engineering systems with signals, timed behaviour, state machines, and reusable components.
The automotive lighting case study fits this style closely, as demonstrated by the vehicle headlight management example.

Medical device interfaces could be represented using dashboards, while railway and elevator controllers could be modelled using Stateflow and displayed using controls, signal plots, or additional visualization components.
The main limitation for the Spectabular workflow is that a model written in another formalism would need to be recreated or connected to the Simulink environment.

### Overall Comparison

VisB and BMotion Studio provide the most direct connection between formal state and arbitrary SVG based domain visualizations.
PVSio-web provides the simplest route to realistic interactive device interfaces.
Spectacle provides the strongest programmatic connection between a formal state and generated SVG graphics.
The Electrum Analyzer provides the closest integration between relational model checking and automatic graph visualization.
Simulink and Stateflow provide the most complete engineering environment for simulation timing, dashboards, signal inspection, and component level execution.

No single approach provides all of the desired features.
The ProB based approaches support formal trace exploration and domain graphics but provide limited support for smooth motion.
PVSio-web provides effective interactive interfaces but limited trace and replay support.
Spectacle supports programmatically generated visuals and trace exploration but requires the visualization to be written as part of the TLA+ environment.
Electrum provides strong automatic analysis views but not arbitrary domain graphics.
Simulink provides mature simulation and timing support but is tied closely to its own modelling ecosystem.

These differences suggest that Spectabular should separate the definition of the visual representation from the control of its execution.
The visualization should remain connected directly to formal state, while interactive exploration, timed execution, scripted scenarios, and replay determine which states are presented and how transitions between them are animated.

## 11. Discussion

### Requirements for Spectabular Visualization

The approaches examined in this report suggest that Spectabular should support two related but distinct forms of visualization: interactive exploration and scripted animation.

Interactive exploration allows the user to directly interact with the current state of a specification.
Graphical controls such as buttons, switches, or selectable objects could trigger operations in the underlying model, after which the visualization would update to reflect the resulting state.
This is the style used by tools such as VisB, BMotion Studio, and PVSio-web, and is useful for exploring a model during development or demonstrating its behaviour to someone unfamiliar with the formal specification.

Scripted animation instead presents a predefined sequence of states or transitions.
The sequence could come from a manually constructed scenario, a saved execution trace, or a counterexample produced during verification.
The animator would step through these states automatically or using playback controls, allowing behaviours to be replayed consistently.
This would be useful for demonstrating scenarios, documenting expected behaviour, replaying counterexamples, and producing animations for reports or presentations.

These two modes should share the same visualization definitions.
A visualization should not need to be rewritten depending on whether the state was reached through direct user interaction or through a prerecorded trace.
In both cases, the visualization should render the current model state, while the animator determines how that state is reached.

The case studies also suggest several practical requirements.
The visualization should support arbitrary domain graphics, movement and changes to graphical properties, reusable controls such as buttons and gauges, multiple objects changing simultaneously, textual and numerical displays, and plots for values that evolve over time.
It should also remain possible to inspect the underlying formal state and execution trace alongside the graphical representation.

These requirements are created by specific limitations observed across the existing approaches.
The ProB based tools provide domain specific SVG visualizations and formal trace exploration but offer limited support for smooth animation between states.
PVSio-web provides effective interactive device prototypes but lacks trace replay and nondeterministic exploration.
Spectacle supports programmatic visualization and trace exploration but requires the visualization to be defined within TLA+.
The Electrum Analyzer generates visualizations automatically from the model structure but does not support arbitrary domain graphics.
Simulink provides mature simulation and timing support but is not designed around formal trace exploration or alternative next states.
Spectabular's combination of interactive exploration and scripted animation over shared SVG visualization definitions, together with its use of Python and Jupyter as the surrounding environment, is intended to address these limitations within a single framework.

### Mapping Table Specifications to Visualizations

A useful starting architecture is similar to VisB, with three main components: the visualization, a mapping between the visualization and the specification, and an animator that manages the current state.

The visualization could be represented using SVG or another graphical format containing identifiable elements.
A mapping would describe how values from the specification affect properties of those elements, such as position, colour, opacity, visibility, text, or other attributes.
The animator would evaluate these mappings whenever the formal state changes and update the visualization accordingly.

Unlike VisB, Spectabular already uses Python as its surrounding language.
The mapping therefore does not necessarily require a separate expression language.
Python expressions and functions could be used directly to transform values from the specification into graphical properties.

For example, a mapping could associate the value of a specification expression with the opacity of a headlight:

```python
{
    "element": "left headlight",
    "attribute": "opacity",
    "value": lambda state: 1 if state["headlights"] else 0
}
```
A more programmatic approach, similar to Spectacle, could also allow the user to define a function that takes the current state and returns an SVG or another graphical representation.
This would be useful for dynamic visualizations where the number or arrangement of graphical objects depends on the specification.

Both approaches could be supported.
Mappings are convenient when an existing SVG or diagram already exists, while programmatic rendering is more flexible for dynamically generated visualizations.

The three types of tables in Spectabular naturally play different roles in this architecture.

Predicate tables describe conditions over the current state.
Their results could directly control graphical properties such as visibility, colour, or whether a control is enabled.
For example, a predicate describing whether the headlights are active could determine whether the corresponding graphical elements are illuminated.

Vector tables naturally produce collections of related values.
These could be useful when several graphical elements need to be updated together.
Rather than defining a separate mapping for every object, a vector table could produce a collection that is then mapped across multiple elements, such as the positions of several vehicles or the states of several sensors.

Relation tables describe transitions from one state to another.
The unprimed variables represent the current state, while the primed variables describe a possible next state.
Animating a relation table therefore requires the animator to determine values for the primed variables and use those values to construct the next state.

### State Transitions and Nondeterminism

Spectabular will need an explicit notion of the current state and the possible transitions from that state.
The animator could maintain the current assignment of specification variables and use relation tables to determine which transitions are currently possible.

For a relation table, the current values of the unprimed variables could be fixed and Z3 could then solve for values of the primed variables.
If no satisfying assignment exists, the transition is not enabled in the current state.
If a satisfying assignment exists, the resulting primed values define a possible next state.

Nondeterminism introduces an additional problem.
A relation may permit several satisfying assignments for the primed variables, meaning that several valid next states may exist.
Simply asking Z3 for one satisfying model would select one of these possibilities arbitrarily and could hide other valid behaviours.

For interactive exploration, Spectabular could expose the available transitions or possible next states and allow the user to choose which one to follow.
This would be similar to ProB displaying enabled operations or Electrum allowing alternative behaviours to be explored.

When there are many possible assignments, presenting every concrete next state may not be practical.
In this case, Spectabular could first present the enabled operation or relation and only ask the user to choose between concrete resulting states when the nondeterminism affects the execution.

For scripted animation, the nondeterminism should already be resolved by the scenario or trace being replayed.
Each step in the script would contain or determine the selected transition and resulting state, allowing the same execution to be reproduced consistently.

### Jupyter Integration

Spectabular is built around Jupyter notebooks, which combine executable Python code, formatted documentation, tables, and visual output within a single document.
This makes Jupyter a natural environment for visualization because the specification, explanation, execution controls, and resulting animation can all be presented together.

A notebook can also serve as a medium for communicating a formal model.
For example, a notebook could be provided to regulators or domain experts containing an explanation of the specification, relevant requirements, predefined scenarios, and a visualization of the system.
A reader could view a scripted animation to inspect expected behaviour or interact directly with the visualization to explore the model.

This is particularly useful for Spectabular because the visualization does not need to become a separate application.
The tabular specification, documentation, execution controls, and domain specific visualization can remain together within the same notebook.

### Candidate Rendering Technologies

Several existing Python and Jupyter technologies could be used to implement the visualization layer.
The relevant capabilities fall into two categories: drawing and animation, which determines how the visualization is rendered and updated, and input and controls, which determines how the user triggers transitions or provides values to the model.
Some libraries address only one of these, while others handle both.

#### Drawing and Animation

| Library / approach | Description | Input | Pros | Cons |
|---|---|---|---|---|
| SVG with JSON Glue + IPython | Display and update SVG directly in the notebook using IPython | Programmatic | Very simple and close to the VisB approach. Existing SVG assets can be reused. | Live updates and callbacks require some JavaScript communication between the browser and Python. |
| ipycanvas | Interactive 2D canvas controlled from Python | Programmatic | Flexible for custom drawing and simple 2D scenes. | The scene generally needs to be drawn programmatically rather than modifying existing SVG elements. |
| [DrawSvg](https://anywidget.dev/en/jupyter-widgets-the-good-parts/) | Programmatic SVG construction and animation from Python | Programmatic | Provides higher level drawing and animation primitives while retaining SVG as the graphical representation. | Animation must be described through explicit keyframes and timings, which becomes difficult to manage as the visualization grows more complex. |

#### Prototype: SVG with JSON Glue + IPython

Demonstrates the VisB-like pattern: an SVG template with named element IDs, a JSON glue mapping that binds formal-state expressions to SVG attributes (fill, y, width, opacity), and `IPython.display` to re-render on each state change.

In [10]:
MAX_FLOOR = 5

def initial_state():
    return {
        "floor": 1,
        "door": "closed",       # closed | open
        "direction": "idle",    # idle | up | down
        "requests": set(),
    }

def request_floor(state, floor):
    s = {**state, "requests": set(state["requests"]) | {floor}}
    return s

def move_to(state, floor):
    s = {**state, "requests": set(state["requests"]) - {floor}}
    if floor > state["floor"]:
        s["direction"] = "up"
    elif floor < state["floor"]:
        s["direction"] = "down"
    s["floor"] = floor
    if not s["requests"]:
        s["direction"] = "idle"
    return s

def open_door(state):
    return {**state, "door": "open"}

def close_door(state):
    return {**state, "door": "closed"}

# trace for replaying
def run_scenario():
    trace = []
    s = initial_state()
    trace.append(("Init", s))

    s = request_floor(s, 4)
    s = request_floor(s, 2)
    trace.append(("Request F2 and F4", s))

    s = move_to(s, 2)
    trace.append(("Move to F2", s))

    s = open_door(s)
    trace.append(("Open door at F2", s))

    s = close_door(s)
    trace.append(("Close door at F2", s))

    s = move_to(s, 4)
    trace.append(("Move to F4", s))

    s = open_door(s)
    trace.append(("Open door at F4", s))

    s = close_door(s)
    trace.append(("Close door, idle at F4", s))

    return trace

from IPython.display import display, HTML
import json as _json

#sameple svg
SVG_TEMPLATE = """
<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 380 340"
     style="font-family:monospace; background:#eceff1; border-radius:10px;">

  <!-- Shaft -->
  <rect id="shaft" x="80" y="30" width="90" height="260" rx="4"
        fill="#cfd8dc" stroke="#90a4ae" stroke-width="1.5"/>

  <!-- Floor markers -->
  {floor_markers}

  <!-- Cabin -->
  <rect id="cabin" x="90" y="{cabin_y}" width="70" height="45" rx="4"
        fill="#455a64" stroke="#263238" stroke-width="2"/>

  <!-- Door (attribute controlled by glue) -->
  <rect id="door" x="{door_x}" y="{door_inner_y}" width="{door_w}" height="35" rx="2"
        fill="{door_fill}"/>
  <text x="125" y="{door_text_y}" text-anchor="middle"
        fill="white" font-size="8">{door_label}</text>

  <!-- Status panel -->
  <rect x="210" y="40" width="150" height="120" rx="8"
        fill="#37474f" stroke="#546e7a"/>
  <text x="285" y="62" text-anchor="middle" fill="#aaa" font-size="10">STATUS</text>
  <text id="floor-display" x="285" y="98" text-anchor="middle"
        fill="white" font-size="32" font-weight="bold">F{floor}</text>
  <text id="dir-display" x="285" y="122" text-anchor="middle"
        fill="{dir_color}" font-size="16">{dir_symbol} {direction}</text>
  <text id="req-display" x="285" y="148" text-anchor="middle"
        fill="#aaa" font-size="9">requests: {requests_txt}</text>

  <!-- Step label -->
  <text x="190" y="322" text-anchor="middle" fill="#555"
        font-size="11" font-weight="bold">{step_label}</text>
</svg>
"""

# sample json glue using python expressions.

GLUE = [
    {"id": "cabin",   "attr": "y",    "expr": lambda s: 30 + (MAX_FLOOR - s["floor"]) * 50},
    {"id": "door",    "attr": "fill", "expr": lambda s: "#a5d6a7" if s["door"]=="open" else "#795548"},
    {"id": "door",    "attr": "width","expr": lambda s: 50 if s["door"]=="open" else 30},
    {"id": "door",    "attr": "x",    "expr": lambda s: 100 if s["door"]=="open" else 110},
]

def apply_glue(state, step_label=""):
    # evaluate the glue expressions for one state -> one static SVG frame
    cabin_y  = GLUE[0]["expr"](state)
    door_fill = GLUE[1]["expr"](state)
    door_w    = GLUE[2]["expr"](state)
    door_x    = GLUE[3]["expr"](state)

    dir_map = {"up": ("▲","#4caf50"), "down": ("▼","#f44336"), "idle": ("●","#9e9e9e")}
    sym, col = dir_map[state["direction"]]
    reqs = ",".join(f"F{r}" for r in sorted(state["requests"])) or "none"

    markers = ""
    for f in range(MAX_FLOOR, 0, -1):
        fy = 30 + (MAX_FLOOR - f) * 50 + 28
        req_fill = "#ff9800" if f in state["requests"] else "#bdbdbd"
        markers += f'<text x="72" y="{fy}" text-anchor="end" fill="#666" font-size="11">F{f}</text>'
        markers += f'<circle cx="178" cy="{fy-5}" r="5" fill="{req_fill}" stroke="#999" stroke-width="0.5"/>'

    # in visb this would just change it live but here we just return a new
    # svg that is our frame
    return SVG_TEMPLATE.format(
        cabin_y=cabin_y, door_fill=door_fill, door_w=door_w, door_x=door_x,
        door_inner_y=cabin_y+5, door_text_y=cabin_y+27, door_label=state["door"],
        floor=state["floor"], direction=state["direction"],
        dir_symbol=sym, dir_color=col, requests_txt=reqs,
        floor_markers=markers, step_label=step_label,
    )

#controls. 
def discrete_stepper(frames, labels, interval_ms=1100):
    frames_json = _json.dumps(frames)
    labels_json = _json.dumps(labels)
    uid = "visb"
    html = f"""
    <div id="visb" style="max-width:420px;margin:auto;font-family:sans-serif;">
      <div class="frame-slot"></div>
      <div style="display:flex;align-items:center;gap:8px;margin-top:6px;">
        <button class="prev">⏮ Prev</button>
        <button class="play">▶ Play</button>
        <button class="next">Next ⏭</button>
        <span class="counter" style="margin-left:auto;font-size:11px;color:#666;"></span>
      </div>
    </div>
    <script>
    (function() {{
        const root = document.currentScript.previousElementSibling;
        const frames = {frames_json};
        const labels = {labels_json};
        let i = 0, timer = null;
        const slot = root.querySelector(".frame-slot");
        const counter = root.querySelector(".counter");
        const playBtn = root.querySelector(".play");
        function show(idx) {{
            i = ((idx % frames.length) + frames.length) % frames.length;
            slot.innerHTML = frames[i];
            counter.textContent = "Step " + (i+1) + " / " + frames.length + " — " + labels[i];
        }}
        function stop() {{
            if (timer) {{ clearInterval(timer); timer = null; playBtn.textContent = "▶ Play"; }}
        }}
        root.querySelector(".prev").addEventListener("click", () => {{ stop(); show(i - 1); }});
        root.querySelector(".next").addEventListener("click", () => {{ stop(); show(i + 1); }});
        playBtn.addEventListener("click", () => {{
            if (timer) {{ stop(); return; }}
            playBtn.textContent = "⏸ Pause";
            timer = setInterval(() => {{
                if (i >= frames.length - 1) {{ stop(); return; }}
                show(i + 1);
            }}, {interval_ms});
        }});
        show(0);
    }})();
    </script>
    """
    display(HTML(html))

print("Prototype 1: SVG with JSON Glue + IPython")

trace = run_scenario()
frames = [apply_glue(state, step_label=action) for action, state in trace]
labels = [action for action, _ in trace]
discrete_stepper(frames, labels)

Prototype 1: SVG with JSON Glue + IPython


#### Prototype: ipycanvas

Demonstrates the canvas pattern: a `Canvas` widget is cleared and redrawn each state using `fill_rect`, `fill_circle`, `fill_text`. Driven by the same real `Play`/`Slider` controls as the ipywidgets prototype — ipycanvas has no interpolation of its own, so each step is a full discrete redraw.

In [2]:
MAX_FLOOR = 5

def initial_state():
    return {
        "floor": 1,
        "door": "closed",       # closed | open
        "direction": "idle",    # idle | up | down
        "requests": set(),
    }

def request_floor(state, floor):
    s = {**state, "requests": set(state["requests"]) | {floor}}
    return s

def move_to(state, floor):
    s = {**state, "requests": set(state["requests"]) - {floor}}
    if floor > state["floor"]:
        s["direction"] = "up"
    elif floor < state["floor"]:
        s["direction"] = "down"
    s["floor"] = floor
    if not s["requests"]:
        s["direction"] = "idle"
    return s

def open_door(state):
    return {**state, "door": "open"}

def close_door(state):
    return {**state, "door": "closed"}

# trace for replaying
def run_scenario():
    trace = []
    s = initial_state()
    trace.append(("Init", s))

    s = request_floor(s, 4)
    s = request_floor(s, 2)
    trace.append(("Request F2 and F4", s))

    s = move_to(s, 2)
    trace.append(("Move to F2", s))

    s = open_door(s)
    trace.append(("Open door at F2", s))

    s = close_door(s)
    trace.append(("Close door at F2", s))

    s = move_to(s, 4)
    trace.append(("Move to F4", s))

    s = open_door(s)
    trace.append(("Open door at F4", s))

    s = close_door(s)
    trace.append(("Close door, idle at F4", s))

    return trace

# ipycanvas, a live Canvas widget, cleared and redrawn each state.
# hold_canvas batches the draw calls into one flush instead of one round
# trip per call. driven by a real Play/Slider, like the ipywidgets prototype.
from ipycanvas import Canvas, hold_canvas
import ipywidgets as widgets
from IPython.display import display

def draw_elevator(canvas, state, label=""):
    with hold_canvas(canvas):
        canvas.clear()
        canvas.fill_style = "#eceff1"
        canvas.fill_rect(0, 0, canvas.width, canvas.height)

        canvas.fill_style = "#cfd8dc"
        canvas.fill_rect(80, 30, 90, 260)
        canvas.stroke_style = "#90a4ae"
        canvas.line_width = 1.5
        canvas.stroke_rect(80, 30, 90, 260)

        canvas.text_align = "end"
        canvas.font = "11px monospace"
        for f in range(MAX_FLOOR, 0, -1):
            fy = 30 + (MAX_FLOOR - f) * 50 + 28
            canvas.fill_style = "#666"
            canvas.fill_text(f"F{f}", 72, fy)
            canvas.fill_style = "#ff9800" if f in state["requests"] else "#bdbdbd"
            canvas.fill_circle(178, fy - 5, 5)

        cabin_y = 30 + (MAX_FLOOR - state["floor"]) * 50
        canvas.fill_style = "#455a64"
        canvas.fill_rect(90, cabin_y, 70, 45)
        canvas.stroke_style = "#263238"
        canvas.line_width = 2
        canvas.stroke_rect(90, cabin_y, 70, 45)

        canvas.fill_style = "#a5d6a7" if state["door"] == "open" else "#795548"
        dw = 50 if state["door"] == "open" else 30
        dx = 100 if state["door"] == "open" else 110
        canvas.fill_rect(dx, cabin_y + 5, dw, 35)
        canvas.text_align = "center"
        canvas.fill_style = "white"
        canvas.font = "8px monospace"
        canvas.fill_text(state["door"], 125, cabin_y + 27)

        canvas.fill_style = "#37474f"
        canvas.fill_rect(210, 40, 150, 120)
        canvas.fill_style = "#aaa"
        canvas.font = "10px monospace"
        canvas.fill_text("STATUS", 285, 62)
        canvas.fill_style = "white"
        canvas.font = "bold 28px monospace"
        canvas.fill_text(f"F{state['floor']}", 285, 98)
        dir_colors = {"up":"#4caf50","down":"#f44336","idle":"#9e9e9e"}
        dir_syms = {"up":"▲","down":"▼","idle":"●"}
        canvas.fill_style = dir_colors[state["direction"]]
        canvas.font = "14px monospace"
        canvas.fill_text(f"{dir_syms[state['direction']]} {state['direction']}", 285, 120)
        canvas.fill_style = "#aaa"
        canvas.font = "9px monospace"
        reqs = ",".join(f"F{r}" for r in sorted(state["requests"])) or "none"
        canvas.fill_text(f"requests: {reqs}", 285, 145)

        canvas.fill_style = "#555"
        canvas.font = "bold 11px monospace"
        canvas.fill_text(label, 190, 322)

print("Prototype 2: ipycanvas (real Canvas widget)")

canvas = Canvas(width=380, height=340)
canvas.layout.border = "1px solid #ddd"

trace = run_scenario()
n = len(trace)

def show_step(idx):
    action, state = trace[idx]
    draw_elevator(canvas, state, label=action)

slider = widgets.IntSlider(min=0, max=n - 1, step=1, value=0, description="Step")
play = widgets.Play(min=0, max=n - 1, step=1, interval=1100, description="Play")
widgets.jslink((play, "value"), (slider, "value"))
slider.observe(lambda change: show_step(change["new"]), names="value")

show_step(0)
display(widgets.VBox([widgets.HBox([play, slider]), canvas]))

Prototype 2: ipycanvas (real Canvas widget)


#### Prototype: DrawSvg

Demonstrates the DrawSvg pattern: SVG elements are built from Python objects (Rectangle, Circle, Text) that carry their own keyframe timelines via `add_key_frame`. drawsvg renders these as native SVG `<animate>` elements, so the browser interpolates continuously between states, with built-in playback controls.

In [3]:
MAX_FLOOR = 5

def initial_state():
    return {
        "floor": 1,
        "door": "closed",       # closed | open
        "direction": "idle",    # idle | up | down
        "requests": set(),
    }

def request_floor(state, floor):
    s = state.copy()
    s["requests"] = set(state["requests"])
    s["requests"].add(floor)
    return s

def move_to(state, floor):
    s = state.copy()
    s["requests"] = set(state["requests"])
    s["requests"].discard(floor)
    if floor > state["floor"]:
        s["direction"] = "up"
    elif floor < state["floor"]:
        s["direction"] = "down"
    s["floor"] = floor
    if not s["requests"]:
        s["direction"] = "idle"
    return s

def open_door(state):
    return {**state, "door": "open"}

def close_door(state):
    return {**state, "door": "closed"}

# trace for replaying
def run_scenario():
    trace = []
    s = initial_state()
    trace.append(("Init", s))

    s = request_floor(s, 4)
    s = request_floor(s, 2)
    trace.append(("Request F2 and F4", s))

    s = move_to(s, 2)
    trace.append(("Move to F2", s))

    s = open_door(s)
    trace.append(("Open door at F2", s))

    s = close_door(s)
    trace.append(("Close door at F2", s))

    s = move_to(s, 4)
    trace.append(("Move to F4", s))

    s = open_door(s)
    trace.append(("Open door at F4", s))

    s = close_door(s)
    trace.append(("Close door, idle at F4", s))

    return trace

# real drawsvg lib. same add_key_frame + SyncedAnimationConfig pattern as
# drawsvg/elevator.py, applied inline to this notebook's own model/trace.
# the browser interpolates continuously between keyframes, so this is one
# playable svg instead of a stack of static frames.
import drawsvg as draw

W, H = 380, 340
SHAFT_X, SHAFT_Y, SHAFT_W, SHAFT_H = 80, 30, 90, 260
CABIN_W, CABIN_H = 70, 45
STATUS_X, STATUS_Y, STATUS_W, STATUS_H = 210, 40, 150, 120

def cabin_y(floor):
    return SHAFT_Y + (MAX_FLOOR - floor) * 50

def floor_marker_y(f):
    return SHAFT_Y + (MAX_FLOOR - f) * 50 + 28

def door_geom(door):
    if door == "open":
        return 100, 50, "#a5d6a7"
    return 110, 30, "#795548"

def build_drawsvg_animation(trace, step_dur=1.3, tail=1.0):
    n = len(trace)
    times = [i * step_dur for i in range(n)]
    duration = times[-1] + tail

    d = draw.Drawing(
        W, H,
        animation_config=draw.types.SyncedAnimationConfig(
            duration=duration,
            show_playback_progress=True,
            show_playback_controls=True,
        ),
    )

    d.append(draw.Rectangle(0, 0, W, H, fill="#eceff1", rx=10))
    d.append(draw.Rectangle(SHAFT_X, SHAFT_Y, SHAFT_W, SHAFT_H, rx=4,
                             fill="#cfd8dc", stroke="#90a4ae", stroke_width=1.5))

    # floor labels + request dots — dot fill is animated, so requests fade in/out
    for f in range(MAX_FLOOR, 0, -1):
        fy = floor_marker_y(f)
        d.append(draw.Text(f"F{f}", 11, SHAFT_X - 8, fy,
                            fill="#666", font_family="monospace", text_anchor="end"))
        dot = draw.Circle(178, fy - 5, 5, fill="#bdbdbd", stroke="#999", stroke_width=0.5)
        for t, (_, s) in zip(times, trace):
            dot.add_key_frame(t, fill=("#ff9800" if f in s["requests"] else "#bdbdbd"))
        d.append(dot)

    # cabin, animated y, glides continuously between floors
    cabin = draw.Rectangle(90, cabin_y(trace[0][1]["floor"]), CABIN_W, CABIN_H, rx=4,
                            fill="#455a64", stroke="#263238", stroke_width=2)
    for t, (_, s) in zip(times, trace):
        cabin.add_key_frame(t, y=cabin_y(s["floor"]))
    d.append(cabin)

    # door, animated x/width/fill, slides and fades open/closed
    dx0, dw0, dc0 = door_geom(trace[0][1]["door"])
    door = draw.Rectangle(dx0, cabin_y(trace[0][1]["floor"]) + 5, dw0, 35, rx=2, fill=dc0)
    for t, (_, s) in zip(times, trace):
        dx, dw, dc = door_geom(s["door"])
        door.add_key_frame(t, x=dx, y=cabin_y(s["floor"]) + 5, width=dw, fill=dc)
    d.append(door)

    # text content can't be tweened, so these are discrete swaps at each keytime
    draw.native_animation.animate_text_sequence(
        d, times, [s["door"] for _, s in trace],
        8, 125, cabin_y(trace[0][1]["floor"]) + 27,
        fill="white", font_family="monospace", text_anchor="middle",
    )

    d.append(draw.Rectangle(STATUS_X, STATUS_Y, STATUS_W, STATUS_H, rx=8,
                             fill="#37474f", stroke="#546e7a"))
    d.append(draw.Text("STATUS", 10, STATUS_X + STATUS_W / 2, STATUS_Y + 22,
                        fill="#aaa", font_family="monospace", text_anchor="middle"))

    draw.native_animation.animate_text_sequence(
        d, times, [f"F{s['floor']}" for _, s in trace],
        32, STATUS_X + STATUS_W / 2, STATUS_Y + 60,
        fill="white", font_weight="bold", font_family="monospace", text_anchor="middle",
    )

    dir_syms = {"up": "▲ up", "down": "▼ down", "idle": "● idle"}
    draw.native_animation.animate_text_sequence(
        d, times, [dir_syms[s["direction"]] for _, s in trace],
        16, STATUS_X + STATUS_W / 2, STATUS_Y + 84,
        fill="#90caf9", font_family="monospace", text_anchor="middle",
    )

    draw.native_animation.animate_text_sequence(
        d, times,
        [("requests: " + (",".join(f"F{r}" for r in sorted(s["requests"])) or "none"))
         for _, s in trace],
        9, STATUS_X + STATUS_W / 2, STATUS_Y + 108,
        fill="#aaa", font_family="monospace", text_anchor="middle",
    )

    draw.native_animation.animate_text_sequence(
        d, times, [action for action, _ in trace],
        11, W / 2, H - 18,
        fill="#555", font_weight="bold", font_family="monospace", text_anchor="middle",
    )

    return d

print("Prototype 3: DrawSvg (real drawsvg lib, native <animate> keyframes)")

trace = run_scenario()
d = build_drawsvg_animation(trace)
d.display_inline()

Prototype 3: DrawSvg (real drawsvg lib, native <animate> keyframes)


JupyterSvgInline(svg='<?xml version="1.0" encoding="UTF-8"?>\n<svg xmlns="http://www.w3.org/2000/svg" xmlns:xlink="http://www.w3.org/1999/xlink"\n     width="380" height="340" viewBox="0 0 380 340" onload="svgOnLoad(event);">\n<defs>\n</defs>\n<rect x="0" y="0" width="380" height="340" fill="#eceff1" rx="10" />\n<rect x="80" y="30" width="90" height="260" rx="4" fill="#cfd8dc" stroke="#90a4ae" stroke-width="1.5" />\n<text x="72" y="58" font-size="11" fill="#666" font-family="monospace" text-anchor="end">F5</text>\n<circle cx="178" cy="53" r="5" fill="#bdbdbd" stroke="#999" stroke-width="0.5">\n<animate attributeName="fill" dur="10.1s" values="#bdbdbd;#bdbdbd;#bdbdbd;#bdbdbd;#bdbdbd;#bdbdbd;#bdbdbd;#bdbdbd;#bdbdbd" keyTimes="0;0.129;0.257;0.386;0.515;0.644;0.772;0.901;1" repeatCount="indefinite" fill="freeze" />\n</circle>\n<text x="72" y="108" font-size="11" fill="#666" font-family="monospace" text-anchor="end">F4</text>\n<circle cx="178" cy="103" r="5" fill="#bdbdbd" stroke="#999" stroke-width="0.5">\n<animate attributeName="fill" dur="10.1s" values="#bdbdbd;#ff9800;#ff9800;#ff9800;#ff9800;#bdbdbd;#bdbdbd;#bdbdbd;#bdbdbd" keyTimes="0;0.129;0.257;0.386;0.515;0.644;0.772;0.901;1" repeatCount="indefinite" fill="freeze" />\n</circle>\n<text x="72" y="158" font-size="11" fill="#666" font-family="monospace" text-anchor="end">F3</text>\n<circle cx="178" cy="153" r="5" fill="#bdbdbd" stroke="#999" stroke-width="0.5">\n<animate attributeName="fill" dur="10.1s" values="#bdbdbd;#bdbdbd;#bdbdbd;#bdbdbd;#bdbdbd;#bdbdbd;#bdbdbd;#bdbdbd;#bdbdbd" keyTimes="0;0.129;0.257;0.386;0.515;0.644;0.772;0.901;1" repeatCount="indefinite" fill="freeze" />\n</circle>\n<text x="72" y="208" font-size="11" fill="#666" font-family="monospace" text-anchor="end">F2</text>\n<circle cx="178" cy="203" r="5" fill="#bdbdbd" stroke="#999" stroke-width="0.5">\n<animate attributeName="fill" dur="10.1s" values="#bdbdbd;#ff9800;#bdbdbd;#bdbdbd;#bdbdbd;#bdbdbd;#bdbdbd;#bdbdbd;#bdbdbd" keyTimes="0;0.129;0.257;0.386;0.515;0.644;0.772;0.901;1" repeatCount="indefinite" fill="freeze" />\n</circle>\n<text x="72" y="258" font-size="11" fill="#666" font-family="monospace" text-anchor="end">F1</text>\n<circle cx="178" cy="253" r="5" fill="#bdbdbd" stroke="#999" stroke-width="0.5">\n<animate attributeName="fill" dur="10.1s" values="#bdbdbd;#bdbdbd;#bdbdbd;#bdbdbd;#bdbdbd;#bdbdbd;#bdbdbd;#bdbdbd;#bdbdbd" keyTimes="0;0.129;0.257;0.386;0.515;0.644;0.772;0.901;1" repeatCount="indefinite" fill="freeze" />\n</circle>\n<rect x="90" y="230" width="70" height="45" rx="4" fill="#455a64" stroke="#263238" stroke-width="2">\n<animate attributeName="y" dur="10.1s" values="230;230;180;180;180;80;80;80;80" keyTimes="0;0.129;0.257;0.386;0.515;0.644;0.772;0.901;1" repeatCount="indefinite" fill="freeze" />\n</rect>\n<rect x="110" y="235" width="30" height="35" rx="2" fill="#795548">\n<animate attributeName="x" dur="10.1s" values="110;110;110;100;110;110;100;110;110" keyTimes="0;0.129;0.257;0.386;0.515;0.644;0.772;0.901;1" repeatCount="indefinite" fill="freeze" />\n<animate attributeName="y" dur="10.1s" values="235;235;185;185;185;85;85;85;85" keyTimes="0;0.129;0.257;0.386;0.515;0.644;0.772;0.901;1" repeatCount="indefinite" fill="freeze" />\n<animate attributeName="width" dur="10.1s" values="30;30;30;50;30;30;50;30;30" keyTimes="0;0.129;0.257;0.386;0.515;0.644;0.772;0.901;1" repeatCount="indefinite" fill="freeze" />\n<animate attributeName="fill" dur="10.1s" values="#795548;#795548;#795548;#a5d6a7;#795548;#795548;#a5d6a7;#795548;#795548" keyTimes="0;0.129;0.257;0.386;0.515;0.644;0.772;0.901;1" repeatCount="indefinite" fill="freeze" />\n</rect>\n<text x="125" y="257" font-size="8" fill="white" font-family="monospace" text-anchor="middle">closed<animate attributeName="visibility" dur="10.1s" values="visible;hidden;hidden" keyTimes="0;0.129;1" repeatCount="indefinite" fill="freeze" /></text>\n<text x="125" y="257" font-size="8" fill="white" font-family="monospace" text-anchor="middle">closed<animate att

#### Input and Controls

| Library / approach | Description | Input | Pros | Cons |
|---|---|---|---|---|
| ipywidgets | Buttons, sliders, dropdowns, play controls, and layout containers | Interactive | Very easy to use in Jupyter and useful for controls around an animation. | Does not provide a drawing or animation system itself. |

This was shown in the ipycanvas example.

#### Both Drawing and Input

Anywidget handles both drawing and input within a single framework, supporting both programmatic and interactive input.
 
| Library / approach | Description | Input | Pros | Cons |
|---|---|---|---|---|
| [Anywidget](https://anywidget.dev/en/jupyter-widgets-the-good-parts/) | Custom Jupyter widgets with synchronized Python and JavaScript state | Both | Provides a clean way to combine SVG, browser interaction, and Python without building the Jupyter communication layer manually. | Requires some JavaScript for the browser side visualization. |

#### Prototype: Anywidget

Demonstrates the recommended Anywidget architecture: Python holds the formal state as traitlets, JavaScript observers watch for `change:state` and update SVG attributes/styles with CSS transitions for smooth animation. The scripted trace replays through one persistent widget, each formal state pushed from Python and eased to in the browser.

In [5]:
MAX_FLOOR = 5

def initial_state():
    return {
        "floor": 1,
        "door": "closed",       # closed | open
        "direction": "idle",    # idle | up | down
        "requests": set(),
    }

def request_floor(state, floor):
    s = {**state, "requests": set(state["requests"]) | {floor}}
    return s

def move_to(state, floor):
    s = {**state, "requests": set(state["requests"]) - {floor}}
    if floor > state["floor"]:
        s["direction"] = "up"
    elif floor < state["floor"]:
        s["direction"] = "down"
    s["floor"] = floor
    if not s["requests"]:
        s["direction"] = "idle"
    return s

def open_door(state):
    return {**state, "door": "open"}

def close_door(state):
    return {**state, "door": "closed"}

# trace for replaying
def run_scenario():
    trace = []
    s = initial_state()
    trace.append(("Init", s))

    s = request_floor(s, 4)
    s = request_floor(s, 2)
    trace.append(("Request F2 and F4", s))

    s = move_to(s, 2)
    trace.append(("Move to F2", s))

    s = open_door(s)
    trace.append(("Open door at F2", s))

    s = close_door(s)
    trace.append(("Close door at F2", s))

    s = move_to(s, 4)
    trace.append(("Move to F4", s))

    s = open_door(s)
    trace.append(("Open door at F4", s))

    s = close_door(s)
    trace.append(("Close door, idle at F4", s))

    return trace

# real anywidget lib. python traitlets hold the state dict, js observes
# "change:state" and updates svg attributes/styles, css transitions do
# the interpolation — python just says what the next state is.
#
# this is a small inline version of the pattern in anywidget/fm.py and
# anywidget/aw.py, rewritten for this notebook's own elevator diagram;
# those examples drive a different diagram (the automotive/pitman-arm svg).
import anywidget
import traitlets
import asyncio
from IPython.display import display

SHAFT_X, SHAFT_Y, SHAFT_W, SHAFT_H = 80, 30, 90, 260
CABIN_W, CABIN_H = 70, 45
STATUS_X, STATUS_Y, STATUS_W, STATUS_H = 210, 40, 150, 120

def _cabin_y(floor):
    return SHAFT_Y + (MAX_FLOOR - floor) * 50

# sets aren't json, so flatten requests to a sorted list before syncing
def _serialize(state):
    # updtste
    return {**state, "requests": sorted(state["requests"])}

def _build_svg():
    base_y = _cabin_y(1)
    markers = ""
    for f in range(MAX_FLOOR, 0, -1):
        fy = SHAFT_Y + (MAX_FLOOR - f) * 50 + 28
        markers += (f'<text x="{SHAFT_X-8}" y="{fy}" text-anchor="end" fill="#666" '
                    f'font-size="11" font-family="monospace">F{f}</text>')
        markers += (f'<circle id="req-{f}" cx="178" cy="{fy-5}" r="5" '
                    f'fill="#bdbdbd" stroke="#999" stroke-width="0.5"/>')
    return f"""
    <svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 380 340"
         style="font-family:monospace;background:#eceff1;border-radius:10px;max-width:420px;">
      <rect x="{SHAFT_X}" y="{SHAFT_Y}" width="{SHAFT_W}" height="{SHAFT_H}" rx="4"
            fill="#cfd8dc" stroke="#90a4ae" stroke-width="1.5"/>
      {markers}
      <g id="cabin-group" style="transition: transform 0.3s ease-in-out;">
        <rect id="cabin" x="90" y="{base_y}" width="{CABIN_W}" height="{CABIN_H}" rx="4"
              fill="#455a64" stroke="#263238" stroke-width="2"/>
        <rect id="door" x="110" y="{base_y+5}" width="30" height="35" rx="2" fill="#795548"
              style="transition: fill 0.3s, width 0.3s, x 0.3s;"/>
        <text id="door-label" x="125" y="{base_y+27}" text-anchor="middle"
              fill="white" font-size="8" font-family="monospace">closed</text>
      </g>
      <rect x="{STATUS_X}" y="{STATUS_Y}" width="{STATUS_W}" height="{STATUS_H}" rx="8"
            fill="#37474f" stroke="#546e7a"/>
      <text x="{STATUS_X+STATUS_W/2}" y="{STATUS_Y+22}" text-anchor="middle"
            fill="#aaa" font-size="10" font-family="monospace">STATUS</text>
      <text id="floor-display" x="{STATUS_X+STATUS_W/2}" y="{STATUS_Y+60}" text-anchor="middle"
            fill="white" font-size="30" font-weight="bold" font-family="monospace">F1</text>
      <text id="dir-display" x="{STATUS_X+STATUS_W/2}" y="{STATUS_Y+84}" text-anchor="middle"
            fill="#9e9e9e" font-size="14" font-family="monospace">● idle</text>
      <text id="req-display" x="{STATUS_X+STATUS_W/2}" y="{STATUS_Y+108}" text-anchor="middle"
            fill="#aaa" font-size="9" font-family="monospace">requests: none</text>
      <text id="step-label" x="190" y="322" text-anchor="middle"
            fill="#555" font-size="11" font-weight="bold" font-family="monospace">Init</text>
    </svg>
    """

ESM = f"""
const MAX_FLOOR = {MAX_FLOOR};
const SHAFT_Y = {SHAFT_Y};

function cabinY(floor) {{ return SHAFT_Y + (MAX_FLOOR - floor) * 50; }}

export default {{
  render({{ model, el }}) {{
    el.innerHTML = model.get("_svg_content");
    const svg = el.querySelector("svg");
    const dirGlyph = {{ up: "▲ up", down: "▼ down", idle: "● idle" }};
    const dirColor = {{ up: "#4caf50", down: "#f44336", idle: "#9e9e9e" }};

    function applyState(state) {{
        const dur = model.get("_transition_duration");
        const cabinGroup = svg.getElementById("cabin-group");
        const door = svg.getElementById("door");
        cabinGroup.style.transitionDuration = dur + "ms";
        door.style.transitionDuration = dur + "ms, " + dur + "ms, " + dur + "ms";

        // cabin: css transform — browser interpolates continuously
        const dy = cabinY(state.floor) - cabinY(1);
        cabinGroup.style.transform = `translateY(${{dy}}px)`;

        // door: css attr transition (fill/x/width are all animatable as css on svg rects)
        if (state.door === "open") {{
            door.style.fill = "#a5d6a7"; door.setAttribute("width", "50"); door.setAttribute("x", "100");
        }} else {{
            door.style.fill = "#795548"; door.setAttribute("width", "30"); door.setAttribute("x", "110");
        }}
        svg.getElementById("door-label").textContent = state.door;

        // discrete text fields — no meaningful interpolation for text content
        svg.getElementById("floor-display").textContent = "F" + state.floor;
        const dirEl = svg.getElementById("dir-display");
        dirEl.textContent = dirGlyph[state.direction];
        dirEl.setAttribute("fill", dirColor[state.direction]);
        const reqTxt = state.requests.length ? state.requests.map(r => "F"+r).join(",") : "none";
        svg.getElementById("req-display").textContent = "requests: " + reqTxt;
        svg.getElementById("step-label").textContent = state.label || "";

        // request dots — css fill transition, continuous fade in/out
        for (let f = 1; f <= MAX_FLOOR; f++) {{
            const dot = svg.getElementById("req-" + f);
            dot.style.transition = "fill " + dur + "ms";
            dot.style.fill = state.requests.includes(f) ? "#ff9800" : "#bdbdbd";
        }}
    }}

    model.on("change:state", () => applyState(model.get("state")));
    applyState(model.get("state"));  // initial render, no animation needed
  }}
}};
"""

class ElevatorFormalWidget(anywidget.AnyWidget):
    _esm = ESM
    _svg_content = traitlets.Unicode("").tag(sync=True)
    _transition_duration = traitlets.Float(0).tag(sync=True)  # ms
    state = traitlets.Dict({}).tag(sync=True)

    def __init__(self, **kw):
        super().__init__(
            _svg_content=_build_svg(),
            state={**_serialize(initial_state()), "label": "Init"},
            **kw,
        )

    async def transition(self, next_state, label="", duration=0.6):
        self._transition_duration = duration * 1000
        self.state = {**_serialize(next_state), "label": label}
        await asyncio.sleep(duration)

print("Prototype 6: Anywidget (traitlets state + CSS-transition observer)")

w = ElevatorFormalWidget()
display(w) 

trace = run_scenario()
for action, state in trace:
    await w.transition(state, label=action, duration=0.6)

Prototype 6: Anywidget (traitlets state + CSS-transition observer)


### Design Direction

The most promising direction is to use Anywidget with SVG as the primary visualization architecture.

DrawSvg was useful for prototyping scripted animations and demonstrated that relatively complex animations can be created inside a Jupyter notebook.
For example, the elevator prototype uses keyframes to animate the cabin, doors, floor indicators, and other components, while the automotive prototype animates the pitman arm and blinking lights.

However, these prototypes also demonstrate a limitation of using DrawSvg as the animator. 
The animation must be described through explicit keyframes and timings for individual graphical elements.
As the visualization becomes more complex, the animation logic becomes closely tied to the graphical implementation and requires a significant amount of code to coordinate transitions between states.

This is not how the animator should work.
The animator should primarily operate in terms of formal states and transitions rather than requiring users to manually construct timelines of SVG keyframes.
For example, a transition from `floor = 1` to `floor = 4` should provide the animator with two formal states, while the visualization layer determines how the elevator moves between their corresponding graphical representations.

Anywidget provides a simpler foundation for this architecture.
An SVG can be embedded directly inside an Anywidget and manipulated using normal JavaScript.
Individual SVG elements can be selected by their identifiers and their attributes, styles, positions, and transforms can be changed directly when the Spectabular state changes.

This is similar to the observer approach used by BMotion Studio.
The Python side of the widget can contain the current formal state, while JavaScript observes changes to that state and updates the corresponding SVG elements.
User interactions with SVG elements can similarly be passed back through the widget to Python, where Spectabular can determine the resulting formal transition.

Animation can then be implemented as part of the visualization layer rather than encoded into the formal model or manually assembled as SVG keyframes.
For example, when the state changes from one elevator floor to another, JavaScript can interpolate the position of the elevator between the graphical representations of those states.
Discrete properties such as a light turning on can be updated immediately, while properties such as position or rotation can be smoothly transitioned when appropriate.

DrawSvg still demonstrates an important feature that should be considered in the final design: animations can be exported as self contained HTML containing the SVG, animation logic, and playback controls.
This is particularly valuable for sharing visualizations outside the Jupyter environment.
A similar export mechanism would allow a completed Spectabular visualization or scripted scenario to be provided to regulators or domain experts as a standalone artifact without requiring them to inspect or execute the underlying specification.

Anywidget with directly embedded SVG therefore appears to provide the simplest foundation for interactive visualization, while the lessons from the DrawSvg prototypes remain useful for designing scripted animation and standalone HTML export.